# Gemma 4 E2B-it Training-Free Weight Surgery & Static-Graph TPU Inference
**Reproducible Research Notebook (Free Google Colab Cloud TPU `v5e-1` / GPU)**

> **License:** Copyright 2026 Keshan Sodimana — Licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0) (see the repository `LICENSE` file).

This self-contained notebook reproduces:
1. **Model Backbone Isolation**: Loading `unsloth/gemma-4-E2B-it` (`Gemma4ForConditionalGeneration`, `bfloat16`) and pruning unused multimodal encoders (`vision_tower`, `audio_tower`) for pure text reasoning/coding.
2. **Phase 1 35-Layer Spectral & Activation MRI**: Measuring `layer_scalar`, PLE gate clamping (`post_per_layer_input_norm`), and KV-shared cosine velocity (`L15-L34`).
3. **`StaticGemma4Engine` (`48x-51x` Colab TPU Speedup)**: Eliminating XLA per-token recompilations via 15-layer static KV allocation (`-57%` KV HBM), device-tensor `index_copy_`, and 4-step block alignment.
4. **Zero-Training Weight Surgeries (`WeightSurgeon`)**:
   - **`S1d` (Code-Safe Prose-Filler Orthogonal PLE Projection)**: Parameter-free projection $W_{\text{proj},\ell}^{\star} = (I - U_{\text{prose}} U_{\text{prose}}^\top) W_{\text{proj},\ell}$ across $\ell \in [25, 34]$.
   - **`S1c` (Surface/Stop Subspace PLE Projection)**: Reasoning-focused subspace attenuation across $\ell \in [25, 34]$.
   - **`S6b` (Scale-Compensated Layer-31 Bypass + Prose PLE Projection)**: Removing dormant Double-Wide MLP Layer 31 (`-56.6M` active parameters) while folding $\lambda_{31}$ into `L30.layer_scalar`.
5. **Paired Statistical Evaluation**: GSM8K, ARC-Challenge, MMLU-Pro (STEM), and MBPP (Sanitized) with exact paired McNemar binomial tests and 2,000-sample bootstrap 95% CIs.


In [ ]:
# @title 1. Setup Workspace Directories & Check Accelerator (TPU v5e-1 or GPU)
import os, torch
os.makedirs("/content/gemma4_surgery/benchmarks", exist_ok=True)
os.makedirs("/content/gemma4_surgery/surgeries", exist_ok=True)
os.makedirs("/content/gemma4_surgery/results", exist_ok=True)
open("/content/gemma4_surgery/benchmarks/__init__.py", "w").close()
open("/content/gemma4_surgery/surgeries/__init__.py", "w").close()

DEVICE = torch.accelerator.current_accelerator() if hasattr(torch, "accelerator") else torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Active Accelerator:", DEVICE, "| PyTorch:", torch.__version__)


In [ ]:
# @title 2. Write Evaluation Harness (`benchmarks/eval_harness.py`)
%%writefile /content/gemma4_surgery/benchmarks/eval_harness.py
# Copyright 2026 Keshan Sodimana
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Rigorous Paired Evaluation Harness for Gemma 4 Weight Surgery Experiments.

Enforces strict Calibration/Dev vs. Held-Out Test separation (zero test leakage),
fixed bucket padding + StaticCache support for fast TPU v5e-1 / GPU batched inference,
and paired statistical testing (McNemar exact binomial test + paired bootstrap 95% CI).
"""

import ast
import json
import math
import os
import random
import re
import subprocess
import sys
import tempfile
import time
from typing import Any, Callable, Dict, List, Optional, Tuple


# ---------------------------------------------------------------------------
# Dataset Loaders (Strict Dev vs. Test Splits)
# ---------------------------------------------------------------------------

def load_gsm8k(split_mode: str = "dev", n: int = 50, seed: int = 42) -> List[Dict[str, Any]]:
    """
    split_mode='dev'  -> samples from GSM8K 'train' split (calibration/ablation only)
    split_mode='test' -> samples from GSM8K 'test' split (held-out final evaluation)
    """
    from datasets import load_dataset

    hf_split = "train" if split_mode == "dev" else "test"
    ds = load_dataset("openai/gsm8k", "main", split=hf_split)
    idxs = list(range(len(ds)))
    rng = random.Random(seed)
    rng.shuffle(idxs)
    chosen = idxs[: min(n, len(idxs))]

    items = []
    for i in chosen:
        row = ds[i]
        ans_raw = row["answer"]
        target = ans_raw.split("####")[-1].strip().replace(",", "")
        items.append({
            "id": f"gsm8k_{hf_split}_{i}",
            "benchmark": "gsm8k",
            "prompt": (
                "Solve the following math word problem concisely in 3 to 5 short lines of arithmetic "
                "(no markdown headers). End your response with exactly 'FINAL ANSWER: <number>'.\n\n"
                f"Problem: {row['question']}"
            ),
            "question_raw": row["question"],
            "target": target,
        })
    return items


def load_arc_challenge(split_mode: str = "dev", n: int = 80, seed: int = 42) -> List[Dict[str, Any]]:
    """
    split_mode='dev'  -> 'validation' split
    split_mode='test' -> 'test' split
    """
    from datasets import load_dataset

    hf_split = "validation" if split_mode == "dev" else "test"
    ds = load_dataset("allenai/ai2_arc", "ARC-Challenge", split=hf_split)
    idxs = list(range(len(ds)))
    rng = random.Random(seed)
    rng.shuffle(idxs)
    chosen = idxs[: min(n, len(idxs))]

    label_map = {"1": "A", "2": "B", "3": "C", "4": "D", "5": "E"}
    items = []
    for i in chosen:
        row = ds[i]
        texts = row["choices"]["text"]
        labels = [label_map.get(l, l) for l in row["choices"]["label"]]
        ans_key = label_map.get(row["answerKey"], row["answerKey"]).strip().upper()
        if len(texts) != 4 or ans_key not in ("A", "B", "C", "D"):
            continue
        opts_str = "\n".join(f"({labels[j]}) {texts[j]}" for j in range(4))
        items.append({
            "id": f"arc_{hf_split}_{i}",
            "benchmark": "arc_challenge",
            "prompt": (
                "Answer the following science question concisely in 2-3 short sentences, then end "
                "with 'FINAL ANSWER: <letter>' where <letter> is A, B, C, or D.\n\n"
                f"Question: {row['question']}\n\n{opts_str}"
            ),
            "choices": dict(zip(labels, texts)),
            "target": ans_key,
        })
    return items


def load_mmlu_pro_stem(split_mode: str = "dev", n: int = 60, seed: int = 42) -> List[Dict[str, Any]]:
    """
    Challenging 10-option graduate/undergraduate STEM reasoning from TIGER-Lab/MMLU-Pro
    (featured directly in the Gemma 4 Technical Report benchmark suite).
    split_mode='dev'  -> 'validation' (70 items) + disjoint slice if needed
    split_mode='test' -> 'test' split filtered to STEM categories (physics, math, chemistry, cs, biology)
    """
    from datasets import load_dataset

    stem_cats = {"math", "physics", "chemistry", "computer science", "biology", "engineering"}
    if split_mode == "dev":
        ds = load_dataset("TIGER-Lab/MMLU-Pro", split="validation")
        pool = [row for row in ds if row.get("category", "").lower() in stem_cats]
        if len(pool) < n:
            # Supplement from tail of test split reserved strictly for dev? No — never touch test!
            # Use full validation split (all categories, 70 items) so test remains 100% untouched.
            pool = list(ds)
    else:
        ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test")
        pool = [row for row in ds if row.get("category", "").lower() in stem_cats]

    rng = random.Random(seed)
    idxs = list(range(len(pool)))
    rng.shuffle(idxs)
    chosen = idxs[: min(n, len(idxs))]

    letters = "ABCDEFGHIJ"
    items = []
    for k in chosen:
        row = pool[k]
        opts = row["options"]
        valid_letters = letters[: len(opts)]
        opts_str = "\n".join(f"({valid_letters[j]}) {opts[j]}" for j in range(len(opts)))
        ans_letter = str(row["answer"]).strip().upper()
        items.append({
            "id": f"mmlu_pro_{split_mode}_{row.get('question_id', k)}",
            "benchmark": "mmlu_pro",
            "category": row.get("category", "unknown"),
            "prompt": (
                "Solve the following multiple-choice problem concisely in 3 to 6 short lines (no verbose headers). "
                f"End your response with 'FINAL ANSWER: <letter>' (one of {', '.join(valid_letters)}).\n\n"
                f"Question: {row['question']}\n\n{opts_str}"
            ),
            "valid_letters": valid_letters,
            "target": ans_letter,
        })
    return items


def load_mbpp_sanitized(split_mode: str = "dev", n: int = 50, seed: int = 42) -> List[Dict[str, Any]]:
    """
    split_mode='dev'  -> MBPP sanitized 'validation' + 'train' splits (never 'test')
    split_mode='test' -> MBPP sanitized 'test' split (257 items)
    """
    from datasets import load_dataset

    if split_mode == "dev":
        ds_val = load_dataset("google-research-datasets/mbpp", "sanitized", split="validation")
        ds_tr = load_dataset("google-research-datasets/mbpp", "sanitized", split="train")
        pool = list(ds_val) + list(ds_tr)
    else:
        pool = list(load_dataset("google-research-datasets/mbpp", "sanitized", split="test"))

    rng = random.Random(seed)
    idxs = list(range(len(pool)))
    rng.shuffle(idxs)
    chosen = idxs[: min(n, len(idxs))]

    items = []
    for i in chosen:
        row = pool[i]
        desc = row.get("prompt") or row.get("text") or ""
        tests = row.get("test_list", [])
        if isinstance(tests, str):
            tests = ast.literal_eval(tests)
        items.append({
            "id": f"mbpp_{split_mode}_{row.get('task_id', i)}",
            "benchmark": "mbpp",
            "prompt": (
                "Write ONLY a self-contained Python 3 solution inside a ```python ... ``` block "
                "(no prose explanation) that satisfies the problem and passes the assertions below.\n\n"
                f"Problem: {desc}\n\nAssertions:\n" + "\n".join(tests[:3])
            ),
            "tests": tests,
            "target": row.get("code", ""),
        })
    return items


# ---------------------------------------------------------------------------
# Deterministic Graders
# ---------------------------------------------------------------------------

def strip_thinking_block(text: str) -> str:
    """Extract final user-facing answer after Gemma 4 thought block / turn delimiters."""
    cleaned = text
    # Strip trailing Gemma 4 turn end markers if decoded with skip_special_tokens=False
    for end_tok in ["<turn|>", "<end_of_turn>", "<eos>"]:
        if end_tok in cleaned:
            cleaned = cleaned.split(end_tok)[0]

    # Native Gemma 4 thinking channel: <|channel>thought\n...\n<channel|>
    if "<channel|>" in cleaned:
        after = cleaned.split("<channel|>")[-1].strip()
        if after:
            return after

    # Legacy / generic thought tags fallback
    for sep in ["<|thought|>", "<thought>"]:
        if sep in cleaned:
            for end_sep in ["<|end_of_thought|>", "</thought>"]:
                if end_sep in cleaned and cleaned.rfind(end_sep) > cleaned.find(sep):
                    after = cleaned.split(end_sep)[-1].strip()
                    if after:
                        return after
    return cleaned.strip()


def grade_gsm8k(output: str, target: str) -> Tuple[bool, Optional[str]]:
    clean = strip_thinking_block(output)
    target_norm = target.strip().replace(",", "")
    try:
        target_val = float(target_norm)
    except ValueError:
        target_val = None

    patterns = [
        r"FINAL\s*ANSWER\s*[:\-]?\s*\$?\s*([\-]?\d[\d,]*(?:\.\d+)?)",
        r"\\boxed\{\s*([\-]?\d[\d,]*(?:\.\d+)?)\s*\}",
        r"(?:answer\s+is|equals|=)\s*\$?\s*([\-]?\d[\d,]*(?:\.\d+)?)\s*[\.\n]*$",
    ]
    pred_str = None
    for pat in patterns:
        matches = re.findall(pat, clean, flags=re.IGNORECASE | re.MULTILINE)
        if matches:
            pred_str = matches[-1].replace(",", "")
            break
    if pred_str is None:
        nums = re.findall(r"[\-]?\d[\d,]*(?:\.\d+)?", clean)
        if nums:
            pred_str = nums[-1].replace(",", "")

    if pred_str is None:
        return False, None
    try:
        if target_val is not None and abs(float(pred_str) - target_val) < 1e-4:
            return True, pred_str
    except ValueError:
        pass
    return (pred_str == target_norm), pred_str


def grade_mcq(output: str, target: str, valid_letters: str = "ABCD") -> Tuple[bool, Optional[str]]:
    clean = strip_thinking_block(output)
    target = target.strip().upper()
    cls_pat = f"[{valid_letters}]"
    patterns = [
        rf"FINAL\s*ANSWER\s*[:\-]?\s*[\(\[]?({cls_pat})[\)\]]?(?![a-zA-Z])",
        rf"\\boxed\{{\s*[\(\[]?({cls_pat})[\)\]]?\s*\}}",
        rf"(?:correct\s+answer|answer)\s*(?:is|:)\s*\**[\(\[]?({cls_pat})[\)\]]?\**(?![a-zA-Z])",
        rf"^\s*[\(\[]?({cls_pat})[\)\]]?\s*$",
    ]
    for pat in patterns:
        matches = re.findall(pat, clean, flags=re.IGNORECASE | re.MULTILINE)
        if matches:
            pred = matches[-1].upper()
            return (pred == target), pred
    # Fallback in last 120 chars
    tail = clean[-120:]
    matches = re.findall(rf"[\(\[]({cls_pat})[\)\]]", tail)
    if matches:
        pred = matches[-1].upper()
        return (pred == target), pred
    return False, None


def grade_mbpp(output: str, tests: List[str], timeout: float = 4.0) -> Tuple[bool, str]:
    clean = strip_thinking_block(output)
    fences = re.findall(r"```(?:python)?\s*(.*?)```", clean, flags=re.DOTALL)
    if fences:
        code = fences[-1].strip()
    else:
        m = re.search(r"((?:import\s+\w+.*\n|from\s+\w+.*\n)*def\s+\w+.*)", clean, flags=re.DOTALL)
        code = m.group(1).strip() if m else clean.strip()
    if "def " not in code:
        return False, "no_def"
    script = code + "\n\n" + "\n".join(tests) + "\n"
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as fh:
        fh.write(script)
        tmp_path = fh.name
    try:
        proc = subprocess.run(
            [sys.executable, tmp_path],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        return (proc.returncode == 0), ("ok" if proc.returncode == 0 else proc.stderr[-160:])
    except subprocess.TimeoutExpired:
        return False, "timeout"
    finally:
        try:
            os.remove(tmp_path)
        except OSError:
            pass


def grade_item(item: Dict[str, Any], output: str) -> Tuple[bool, Any]:
    b = item["benchmark"]
    if b == "gsm8k":
        return grade_gsm8k(output, item["target"])
    elif b in ("arc_challenge", "mmlu_pro"):
        return grade_mcq(output, item["target"], item.get("valid_letters", "ABCD"))
    elif b == "mbpp":
        return grade_mbpp(output, item["tests"])
    raise ValueError(f"Unknown benchmark {b}")


# ---------------------------------------------------------------------------
# Paired Statistical Tests (McNemar Exact + Paired Bootstrap CI)
# ---------------------------------------------------------------------------

def paired_significance(base_correct: List[bool], surg_correct: List[bool], seed: int = 42) -> Dict[str, Any]:
    assert len(base_correct) == len(surg_correct)
    n = len(base_correct)
    b01 = sum((not b) and s for b, s in zip(base_correct, surg_correct))  # surgery fixed baseline error
    b10 = sum(b and (not s) for b, s in zip(base_correct, surg_correct))  # surgery broke baseline success
    b11 = sum(b and s for b, s in zip(base_correct, surg_correct))
    b00 = sum((not b) and (not s) for b, s in zip(base_correct, surg_correct))

    # Exact two-sided McNemar binomial p-value on discordant pairs
    disc = b01 + b10
    if disc == 0:
        p_val = 1.0
    else:
        k = min(b01, b10)
        tail = sum(math.comb(disc, i) for i in range(k + 1)) / (2 ** disc)
        p_val = min(1.0, 2.0 * tail)

    # Paired bootstrap 95% CI on accuracy delta
    rng = random.Random(seed)
    deltas = [int(s) - int(b) for b, s in zip(base_correct, surg_correct)]
    boot = []
    for _ in range(2000):
        samp = [deltas[rng.randrange(n)] for _ in range(n)]
        boot.append(sum(samp) / n)
    boot.sort()
    return {
        "n": n,
        "base_acc": round(sum(base_correct) / max(1, n), 4),
        "surg_acc": round(sum(surg_correct) / max(1, n), 4),
        "delta_acc": round((sum(surg_correct) - sum(base_correct)) / max(1, n), 4),
        "gains_b01": b01,
        "losses_b10": b10,
        "both_correct_b11": b11,
        "both_wrong_b00": b00,
        "mcnemar_exact_p": round(p_val, 5),
        "ci95_delta": [round(boot[int(0.025 * len(boot))], 4), round(boot[int(0.975 * len(boot))], 4)],
    }


In [ ]:
# @title 3. Write Static-Graph Pure-Backbone Engine (`benchmarks/fast_infer.py`)
%%writefile /content/gemma4_surgery/benchmarks/fast_infer.py
# Copyright 2026 Keshan Sodimana
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Static-Graph Pure-Backbone Batched Inference Engine for Gemma 4 (`Gemma4TextModel` + `lm_head`).

Eliminates two major bottlenecks observed in default `Gemma4ForConditionalGeneration.generate`:
1. Bypasses outer multimodal wrapper overhead (`get_placeholder_mask` + `[B, S, 1536]` clones).
2. Replaces growing `DynamicCache` and Python-int slice constants with:
   - Pre-allocated KV buffers ONLY for the 15 KV-computing layers (`L0..L14`), saving 57% KV HBM.
   - Device-tensor `index_copy_(2, cur_pos_tensor, ...)` updates and tensor-derived 4D masks so
     every decode step `s = 0 .. max_new_tokens - 1` produces a 100% identical HLO graph on TPU/XLA!
"""

import time
from typing import Any, Dict, List, Optional, Sequence, Tuple
import torch
import torch.nn as nn


def get_step_barrier(device: torch.device):
    """Return per-step XLA graph boundary function on TPU, or no-op on CUDA/CPU."""
    if device.type == "tpu":
        return torch.accelerator.synchronize
    elif device.type == "xla":
        import torch_xla.core.xla_model as xm  # type: ignore

        return xm.mark_step
    return lambda: None


class Gemma4StaticKVCache:
    """
    Shape-invariant KV cache allocating buffers ONLY for non-KV-shared layers (`L0..L14`).
    Uses device-tensor `index_copy_` during decode so XLA compiles the decode step ONCE.
    """

    def __init__(
        self,
        text_config: Any,
        batch_size: int,
        max_total_len: int,
        device: torch.device,
        dtype: torch.dtype = torch.bfloat16,
    ):
        self.batch_size = batch_size
        self.max_total_len = max_total_len
        self.device = device
        self.dtype = dtype
        self.first_kv_shared = text_config.num_hidden_layers - getattr(text_config, "num_kv_shared_layers", 0)
        self.k_buf: Dict[int, torch.Tensor] = {}
        self.v_buf: Dict[int, torch.Tensor] = {}

        for l in range(self.first_kv_shared):
            l_cfg = text_config.per_layer_config[l]
            kv_heads = l_cfg.num_key_value_heads
            head_dim = l_cfg.head_dim
            self.k_buf[l] = torch.zeros(
                (batch_size, kv_heads, max_total_len, head_dim), device=device, dtype=dtype
            )
            self.v_buf[l] = torch.zeros(
                (batch_size, kv_heads, max_total_len, head_dim), device=device, dtype=dtype
            )

        self.mode: str = "prefill"
        self.prefill_positions: Optional[torch.Tensor] = None
        self.decode_pos: Optional[torch.Tensor] = None  # shape [1], int64 on device

    def reset_zero_(self) -> None:
        for l in self.k_buf:
            self.k_buf[l].zero_()
            self.v_buf[l].zero_()

    def set_prefill(self, prefill_positions: torch.Tensor) -> None:
        """`prefill_positions`: 1D int64 device tensor `arange(P)`."""
        self.mode = "prefill"
        self.prefill_positions = prefill_positions
        self.decode_pos = None

    def set_decode(self, decode_pos: torch.Tensor) -> None:
        """`decode_pos`: 1D int64 device tensor of shape `[1]`."""
        self.mode = "decode"
        self.decode_pos = decode_pos

    def get_seq_length(self, layer_idx: int = 0) -> int:
        return 0

    def update(
        self,
        key_states: torch.Tensor,
        value_states: torch.Tensor,
        layer_idx: int,
        cache_kwargs: Optional[Dict[str, Any]] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        kb = self.k_buf[layer_idx]
        vb = self.v_buf[layer_idx]
        if self.mode == "prefill":
            assert self.prefill_positions is not None
            kb.index_copy_(2, self.prefill_positions, key_states)
            vb.index_copy_(2, self.prefill_positions, value_states)
            return key_states, value_states
        else:
            assert self.decode_pos is not None
            kb.index_copy_(2, self.decode_pos, key_states)
            vb.index_copy_(2, self.decode_pos, value_states)
            return kb, vb


class StaticGemma4Engine:
    """High-throughput deterministic greedy generation engine on Gemma 4's pure text backbone."""

    def __init__(
        self,
        cond_or_causal_model: nn.Module,
        tokenizer: Any,
        device: torch.device,
    ):
        self.outer_model = cond_or_causal_model
        if hasattr(cond_or_causal_model, "model") and hasattr(cond_or_causal_model.model, "language_model"):
            self.text_model = cond_or_causal_model.model.language_model
        elif hasattr(cond_or_causal_model, "model"):
            self.text_model = cond_or_causal_model.model
        else:
            raise ValueError("Unsupported Gemma 4 wrapper structure")

        self.lm_head = cond_or_causal_model.lm_head
        self.config = self.text_model.config
        self.tokenizer = tokenizer
        self.device = device
        self.dtype = next(self.text_model.parameters()).dtype
        self.softcap: Optional[float] = getattr(self.config, "final_logit_softcapping", None)
        self.sliding_window: int = int(getattr(self.config, "sliding_window", 512) or 512)
        self.pad_token_id: int = int(getattr(self.config, "pad_token_id", 0) or 0)
        self.mark_step = get_step_barrier(device)

        # Collect all stop token IDs (<eos>, <turn|>, <end_of_turn>)
        stop_ids = set()
        eos_cfg = getattr(self.config, "eos_token_id", None)
        if isinstance(eos_cfg, int):
            stop_ids.add(eos_cfg)
        elif isinstance(eos_cfg, (list, tuple)):
            stop_ids.update(int(x) for x in eos_cfg)
        for tok_str in ["<eos>", "<turn|>", "<end_of_turn>"]:
            tid = tokenizer.convert_tokens_to_ids(tok_str)
            if isinstance(tid, int) and tid >= 0 and tid != tokenizer.unk_token_id:
                stop_ids.add(tid)
        self.stop_ids_list = sorted(stop_ids)
        self.stop_ids_dev = torch.tensor(self.stop_ids_list, device=device, dtype=torch.long)

        # Reusable KV cache pool keyed by (batch_size, max_total_len)
        self._cache_pool: Dict[Tuple[int, int], Gemma4StaticKVCache] = {}

    def _get_cache(self, batch_size: int, max_total_len: int) -> Gemma4StaticKVCache:
        key = (batch_size, max_total_len)
        if key not in self._cache_pool:
            # Keep pool bounded to avoid HBM fragmentation
            if len(self._cache_pool) >= 2:
                self._cache_pool.clear()
            self._cache_pool[key] = Gemma4StaticKVCache(
                self.config, batch_size, max_total_len, self.device, self.dtype
            )
        cache = self._cache_pool[key]
        cache.reset_zero_()
        return cache

    def _head_argmax(self, last_hidden: torch.Tensor) -> torch.Tensor:
        """
        Compute greedy next token `[B, 1]` from `last_hidden` `[B, 1, D]`.
        Note: `tanh(x / c) * c` is strictly monotonic increasing on (-inf, +inf), so
        `argmax(softcap(logits)) == argmax(logits)` identically while saving a 262k tanh!
        """
        logits = self.lm_head(last_hidden)  # [B, 1, V]
        return logits.argmax(dim=-1)  # [B, 1]

    def format_chat_prompts(
        self,
        user_prompts: Sequence[str],
        enable_thinking: bool = False,
        system_prompt: Optional[str] = None,
    ) -> List[str]:
        formatted = []
        for p in user_prompts:
            msgs = []
            if system_prompt:
                msgs.append({"role": "system", "content": system_prompt})
            msgs.append({"role": "user", "content": p})
            txt = self.tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=enable_thinking,
            )
            formatted.append(txt)
        return formatted

    @torch.no_grad()
    def generate_batch_static(
        self,
        formatted_prompts: Sequence[str],
        max_new_tokens: int = 256,
        prompt_pad_bucket: int = 256,
        early_stop_check_every: int = 16,
        sync_every: int = 4,
    ) -> Dict[str, Any]:
        """
        Fixed-shape bucketed greedy generation for a single batch of `B` formatted strings.
        All tensors inside the decode loop have invariant shapes `[B, 1]` and `[B, 1, M, d]`.
        """
        B = len(formatted_prompts)
        enc = [self.tokenizer.encode(t, add_special_tokens=False) for t in formatted_prompts]
        max_p_actual = max(len(ids) for ids in enc)
        # Round prompt_pad_len up to a multiple of prompt_pad_bucket so shapes stay bucketed
        P = max(prompt_pad_bucket, ((max_p_actual + prompt_pad_bucket - 1) // prompt_pad_bucket) * prompt_pad_bucket)
        sync_k = max(1, int(sync_every))
        # Align (G - 1) to an exact multiple of sync_k so every decode block has identical HLO shape!
        rem_steps = max(0, max_new_tokens - 1)
        aligned_rem = ((rem_steps + sync_k - 1) // sync_k) * sync_k
        G = 1 + aligned_rem
        M = P + G

        input_cpu = torch.full((B, P), self.pad_token_id, dtype=torch.long)
        prompt_lens_cpu = torch.zeros((B,), dtype=torch.long)
        for i, ids in enumerate(enc):
            if len(ids) > P:
                ids = ids[-P:]
            L = len(ids)
            input_cpu[i, P - L : P] = torch.tensor(ids, dtype=torch.long)
            prompt_lens_cpu[i] = L

        t_start = time.perf_counter()
        input_ids = input_cpu.to(self.device)
        prompt_lens = prompt_lens_cpu.to(self.device)  # [B]
        valid_start = (P - prompt_lens)[:, None, None, None]  # [B, 1, 1, 1]

        cache = self._get_cache(B, M)
        zero_bf16 = torch.zeros((), device=self.device, dtype=self.dtype)
        neg_inf_bf16 = torch.full((), torch.finfo(self.dtype).min, device=self.device, dtype=self.dtype)

        # ---- 1. Static-Shape Prefill ([B, P]) ----
        prefill_arange = torch.arange(P, device=self.device, dtype=torch.long)
        q_idx = prefill_arange[None, None, :, None]  # [1, 1, P, 1]
        k_idx_p = prefill_arange[None, None, None, :]  # [1, 1, 1, P]

        is_pad_q = q_idx < valid_start
        pad_self = is_pad_q & (k_idx_p == q_idx)
        real_causal = (~is_pad_q) & (k_idx_p >= valid_start) & (k_idx_p <= q_idx)

        allow_full_p = pad_self | real_causal
        allow_slid_p = pad_self | (real_causal & ((q_idx - k_idx_p) < self.sliding_window))

        prefill_masks = {
            "full_attention": torch.where(allow_full_p, zero_bf16, neg_inf_bf16),
            "sliding_attention": torch.where(allow_slid_p, zero_bf16, neg_inf_bf16),
        }
        prefill_pos_ids = (q_idx.squeeze(1).squeeze(-1) - valid_start.squeeze(-1).squeeze(-1)).clamp(min=0)  # [B, P]

        cache.set_prefill(prefill_arange)
        out_p = self.text_model(
            input_ids=input_ids,
            attention_mask=prefill_masks,
            position_ids=prefill_pos_ids,
            past_key_values=cache,
            use_cache=True,
        )
        next_tok = self._head_argmax(out_p.last_hidden_state[:, -1:, :])  # [B, 1]

        gen_tokens = torch.full((B, G), self.pad_token_id, device=self.device, dtype=torch.long)
        finished = torch.zeros((B, 1), device=self.device, dtype=torch.bool)

        gen_tokens[:, 0:1] = next_tok
        finished = finished | torch.isin(next_tok, self.stop_ids_dev)
        self.mark_step()

        # ---- 2. Shape-Invariant Decode Loop ([B, 1] against [B, 1, M, d]) ----
        k_idx_m = torch.arange(M, device=self.device, dtype=torch.long)[None, None, None, :]  # [1, 1, 1, M]
        # Pre-materialize step index tensors on device so zero Python ints enter HLO!
        step_cache_positions = torch.arange(P, P + G, device=self.device, dtype=torch.long).unsqueeze(1)  # [G, 1]
        step_offsets = torch.arange(0, G, device=self.device, dtype=torch.long).unsqueeze(1)  # [G, 1]
        pad_tok_tensor = torch.full((B, 1), self.pad_token_id, device=self.device, dtype=torch.long)

        steps_run = 1
        for s in range(G - 1):
            cur_pos_1d = step_cache_positions[s]  # shape [1] on device
            cur_pos_4d = cur_pos_1d.view(1, 1, 1, 1)
            dec_pos_ids = prompt_lens[:, None] + step_offsets[s]  # [B, 1] on device

            allow_full_d = (k_idx_m >= valid_start) & (k_idx_m <= cur_pos_4d)
            allow_slid_d = allow_full_d & ((cur_pos_4d - k_idx_m) < self.sliding_window)
            decode_masks = {
                "full_attention": torch.where(allow_full_d, zero_bf16, neg_inf_bf16),
                "sliding_attention": torch.where(allow_slid_d, zero_bf16, neg_inf_bf16),
            }

            cache.set_decode(cur_pos_1d)
            out_d = self.text_model(
                input_ids=next_tok,
                attention_mask=decode_masks,
                position_ids=dec_pos_ids,
                past_key_values=cache,
                use_cache=True,
            )
            cand_tok = self._head_argmax(out_d.last_hidden_state)  # [B, 1]
            next_tok = torch.where(finished, pad_tok_tensor, cand_tok)
            # Device-indexed write into gen_tokens[:, s+1] without Python slice constant if desired:
            write_col = step_offsets[s + 1]  # shape [1]
            gen_tokens.index_copy_(1, write_col, next_tok)
            finished = finished | torch.isin(next_tok, self.stop_ids_dev)
            steps_run += 1

            if (s + 1) % sync_k == 0:
                self.mark_step()
                if early_stop_check_every > 0 and ((s + 1) % early_stop_check_every == 0):
                    if bool(finished.all().cpu().item()):
                        break

        if self.device.type == "tpu" and hasattr(torch.accelerator, "synchronize"):
            torch.accelerator.synchronize()
        elif self.device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t_start

        actual_limit = min(steps_run, max_new_tokens)
        gen_cpu = gen_tokens[:, :actual_limit].cpu().tolist()
        stop_set = set(self.stop_ids_list)
        texts: List[str] = []
        tok_counts: List[int] = []
        for row in gen_cpu:
            trimmed = []
            for tid in row:
                if tid in stop_set:
                    break
                trimmed.append(tid)
            tok_counts.append(len(trimmed))
            texts.append(self.tokenizer.decode(trimmed, skip_special_tokens=False))

        total_gen_toks = sum(tok_counts)
        return {
            "texts": texts,
            "token_counts": tok_counts,
            "total_generated_tokens": total_gen_toks,
            "steps_executed": actual_limit,
            "batch_size": B,
            "prompt_pad_len": P,
            "elapsed_sec": round(elapsed, 4),
            "effective_tok_per_sec": round(total_gen_toks / max(elapsed, 1e-6), 2),
            "raw_step_tok_per_sec": round((B * actual_limit) / max(elapsed, 1e-6), 2),
        }

    def generate_dataset(
        self,
        prompts: Sequence[str],
        batch_size: int = 32,
        max_new_tokens: int = 256,
        prompt_pad_bucket: int = 256,
        enable_thinking: bool = False,
        system_prompt: Optional[str] = None,
        sync_every: int = 4,
    ) -> Dict[str, Any]:
        """Run batched generation across an entire dataset, padding tail batch for shape reuse."""
        formatted = self.format_chat_prompts(
            prompts, enable_thinking=enable_thinking, system_prompt=system_prompt
        )
        all_texts: List[str] = []
        all_counts: List[int] = []
        total_elapsed = 0.0

        n = len(formatted)
        for start in range(0, n, batch_size):
            chunk = list(formatted[start : start + batch_size])
            real_len = len(chunk)
            if real_len < batch_size:
                # Pad tail batch with copies of last prompt so batch_size stays identical (zero XLA recompile!)
                chunk.extend([chunk[-1]] * (batch_size - real_len))
            res = self.generate_batch_static(
                chunk,
                max_new_tokens=max_new_tokens,
                prompt_pad_bucket=prompt_pad_bucket,
                sync_every=sync_every,
            )
            all_texts.extend(res["texts"][:real_len])
            all_counts.extend(res["token_counts"][:real_len])
            total_elapsed += res["elapsed_sec"]

        tot_toks = sum(all_counts)
        return {
            "texts": all_texts,
            "token_counts": all_counts,
            "total_tokens": tot_toks,
            "total_elapsed_sec": round(total_elapsed, 3),
            "tok_per_sec": round(tot_toks / max(total_elapsed, 1e-6), 2),
        }


In [ ]:
# @title 4. Write Reversible Weight Surgeon (`surgeries/weight_surgeon.py`)
%%writefile /content/gemma4_surgery/surgeries/weight_surgeon.py
# Copyright 2026 Keshan Sodimana
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Reversible, In-Place Weight & Structural Surgery Toolkit for Gemma 4.

Supports:
1. Bit-exact host-RAM backups and restoration for BOTH `nn.Parameter` and `nn.Buffer`
   (critical because Gemma 4's `layers.{l}.layer_scalar` is an `nn.Buffer`!).
2. PLE De-Clamping & Lexical Attenuation (`post_per_layer_input_norm`, `embed_tokens_per_layer`).
3. KV-Shared Query Inverse-Temperature & Context-Retrieval Gain Calibration
   (`self_attn.q_norm.weight`, `post_attention_layernorm.weight`).
4. Spectral SVD Rank-Filtering (`per_layer_projection`, `mlp.down_proj`, `self_attn.o_proj`).
5. Scale-Compensated KV-Shared Layer Bypass (`BypassDecoderLayer` + cumulative `layer_scalar`
   grafting), safely eliminating redundant upper Double-Wide MLP + shared-KV layers.
6. Rank-k Subspace Projection / Contrastive Direction Grafting directly into projection weights.
"""

from typing import Dict, Iterable, List, Optional
import torch
import torch.nn as nn


class BypassDecoderLayer(nn.Module):
    """Zero-FLOP pass-through replacement for redundant KV-shared Gemma 4 decoder layers."""

    def __init__(self, orig_layer: nn.Module):
        super().__init__()
        self.layer_idx = getattr(orig_layer, "layer_idx", -1)
        # Preserve attribute interface expected by Gemma4TextModel utilities
        self.self_attn = orig_layer.self_attn

    def forward(self, hidden_states: torch.Tensor, *args, **kwargs) -> torch.Tensor:
        return hidden_states


class WeightSurgeon:
    """Manages bit-exact backups in host RAM and applies composable, reversible surgeries."""

    def __init__(self, text_model: nn.Module):
        self.text_model = text_model
        self.tensor_map: Dict[str, torch.Tensor] = {}
        for k, p in text_model.named_parameters():
            self.tensor_map[k] = p
        for k, b in text_model.named_buffers():
            self.tensor_map[k] = b

        self.backups: Dict[str, torch.Tensor] = {}
        self.layer_module_backups: Dict[int, nn.Module] = {}
        self.active_log: List[str] = []

    def _ensure_backup(self, name: str) -> None:
        if name not in self.backups:
            t = self.tensor_map[name]
            self.backups[name] = t.detach().to(device="cpu").clone()

    def restore_all(self) -> int:
        """Bit-exact restoration of all modified parameters, buffers, and bypassed layers."""
        # First restore any swapped decoder layer modules
        for idx, orig_mod in self.layer_module_backups.items():
            self.text_model.layers[idx] = orig_mod
        restored_layers = len(self.layer_module_backups)
        self.layer_module_backups.clear()

        # Refresh tensor_map in case modules were restored
        self.tensor_map.clear()
        for k, p in self.text_model.named_parameters():
            self.tensor_map[k] = p
        for k, b in self.text_model.named_buffers():
            self.tensor_map[k] = b

        count = 0
        with torch.no_grad():
            for name, orig_cpu in self.backups.items():
                t = self.tensor_map[name]
                t.data.copy_(orig_cpu.to(device=t.device, dtype=t.dtype))
                count += 1
        self.active_log.clear()
        return count + restored_layers

    # ------------------------------------------------------------------
    # S1. PLE Post-Norm Gain Schedule Surgery (Hypothesis H1)
    # ------------------------------------------------------------------
    def scale_ple_post_norm(self, layers: Iterable[int], scale: float | Dict[int, float]) -> None:
        """
        Rescale `layers.{l}.post_per_layer_input_norm.weight` (Gemma4RMSNorm uses direct `w` scaling)
        to attenuate clamped static Layer-0 PLE injections relative to contextual updates.
        """
        layers_list = list(layers)
        with torch.no_grad():
            for l in layers_list:
                alpha = scale[l] if isinstance(scale, dict) else float(scale)
                name = f"layers.{l}.post_per_layer_input_norm.weight"
                if name not in self.tensor_map:
                    continue
                self._ensure_backup(name)
                orig = self.backups[name].float()
                t = self.tensor_map[name]
                t.data.copy_((orig * alpha).to(device=t.device, dtype=t.dtype))
        self.active_log.append(f"scale_ple_post_norm(layers={layers_list}, scale={scale})")

    # ------------------------------------------------------------------
    # S2a. KV-Shared Query-Norm Inverse-Temperature Surgery (Hypothesis H2)
    # ------------------------------------------------------------------
    def scale_q_norm_temperature(
        self,
        layers: Iterable[int],
        temp_scale: float | Dict[int, float],
    ) -> None:
        """
        Modulate exact attention inverse-temperature in target layers via `self_attn.q_norm.weight`.
        Because Gemma4TextAttention sets `self.scaling = 1.0` and Gemma4RMSNorm multiplies directly
        by `weight`, scaling `q_norm.weight` by `tau` multiplies QK^T logits by exact factor `tau`.
        """
        layers_list = list(layers)
        with torch.no_grad():
            for l in layers_list:
                tau = temp_scale[l] if isinstance(temp_scale, dict) else float(temp_scale)
                name = f"layers.{l}.self_attn.q_norm.weight"
                if name not in self.tensor_map:
                    continue
                self._ensure_backup(name)
                orig = self.backups[name].float()
                t = self.tensor_map[name]
                t.data.copy_((orig * tau).to(device=t.device, dtype=t.dtype))
        self.active_log.append(f"scale_q_norm_temperature(layers={layers_list}, tau={temp_scale})")

    # ------------------------------------------------------------------
    # S2b. Contextual Attention Retrieval Gain Surgery
    # ------------------------------------------------------------------
    def scale_post_attn_norm(
        self,
        layers: Iterable[int],
        scale: float | Dict[int, float],
    ) -> None:
        """
        Modulate `layers.{l}.post_attention_layernorm.weight` to strengthen or rebalance
        contextual attention updates against static PLE / MLP updates in KV-shared layers.
        """
        layers_list = list(layers)
        with torch.no_grad():
            for l in layers_list:
                alpha = scale[l] if isinstance(scale, dict) else float(scale)
                name = f"layers.{l}.post_attention_layernorm.weight"
                if name not in self.tensor_map:
                    continue
                self._ensure_backup(name)
                orig = self.backups[name].float()
                t = self.tensor_map[name]
                t.data.copy_((orig * alpha).to(device=t.device, dtype=t.dtype))
        self.active_log.append(f"scale_post_attn_norm(layers={layers_list}, scale={scale})")

    # ------------------------------------------------------------------
    # S3. Spectral SVD Rank-Filtering Surgery (Hypothesis H3)
    # ------------------------------------------------------------------
    def svd_filter_matrix(
        self,
        layers: Iterable[int],
        module_suffix: str,
        keep_ratio: float = 0.5,
        energy_preserve: bool = True,
    ) -> None:
        """
        Truncate trailing singular values of `layers.{l}.{module_suffix}.weight`
        (e.g. `per_layer_projection`, `mlp.down_proj`, `self_attn.o_proj`) to top `keep_ratio`.
        """
        layers_list = list(layers)
        with torch.no_grad():
            for l in layers_list:
                name = f"layers.{l}.{module_suffix}.weight"
                if name not in self.tensor_map:
                    continue
                self._ensure_backup(name)
                w = self.backups[name].float()
                u, s, vh = torch.linalg.svd(w, full_matrices=False)
                k = max(1, int(round(len(s) * keep_ratio)))
                s_trunc = s.clone()
                s_trunc[k:] = 0.0
                if energy_preserve:
                    orig_norm = s.norm()
                    new_norm = s_trunc.norm()
                    if new_norm > 1e-8:
                        s_trunc = s_trunc * (orig_norm / new_norm)
                w_new = (u * s_trunc.unsqueeze(0)) @ vh
                t = self.tensor_map[name]
                t.data.copy_(w_new.to(device=t.device, dtype=t.dtype))
        self.active_log.append(
            f"svd_filter_matrix(layers={layers_list}, suffix={module_suffix}, keep_ratio={keep_ratio}, energy={energy_preserve})"
        )

    # ------------------------------------------------------------------
    # S4. Scale-Compensated KV-Shared Layer Bypass Surgery
    # ------------------------------------------------------------------
    def bypass_kv_shared_layers(
        self,
        bypass_layers: Iterable[int],
        compensate_scalar: bool = True,
    ) -> Dict[str, float]:
        """
        Bypass specified KV-shared layers (`l >= 15`) using `BypassDecoderLayer`.
        When `compensate_scalar=True`, folds each bypassed block's `layer_scalar` (`lambda_l`)
        multiplicatively into the nearest preceding non-bypassed layer's `layer_scalar` buffer
        so downstream residual stream norm matches the unpruned trajectory!
        """
        bypass_set = sorted(set(bypass_layers))
        for l in bypass_set:
            if l < 15:
                raise ValueError(f"Layer {l} is NOT a KV-shared layer (must be >= 15 to preserve KV anchors).")

        applied_scales: Dict[str, float] = {}
        with torch.no_grad():
            if compensate_scalar:
                # Accumulate scalar product onto nearest preceding active layer
                accum: Dict[int, float] = {}
                for l in bypass_set:
                    sc_name = f"layers.{l}.layer_scalar"
                    self._ensure_backup(sc_name)
                    lam = float(self.backups[sc_name].float().item())
                    prev = l - 1
                    while prev in bypass_set and prev >= 0:
                        prev -= 1
                    if prev < 0:
                        raise ValueError("Cannot bypass Layer 0.")
                    accum[prev] = accum.get(prev, 1.0) * lam

                for prev_l, mult in accum.items():
                    prev_name = f"layers.{prev_l}.layer_scalar"
                    self._ensure_backup(prev_name)
                    orig_prev = self.backups[prev_name].float()
                    t = self.tensor_map[prev_name]
                    t.data.copy_((orig_prev * mult).to(device=t.device, dtype=t.dtype))
                    applied_scales[f"layer_{prev_l}_mult"] = round(mult, 6)

            for l in bypass_set:
                if l not in self.layer_module_backups:
                    self.layer_module_backups[l] = self.text_model.layers[l]
                self.text_model.layers[l] = BypassDecoderLayer(self.layer_module_backups[l])

        self.active_log.append(
            f"bypass_kv_shared_layers(layers={bypass_set}, compensate_scalar={compensate_scalar}, scales={applied_scales})"
        )
        return applied_scales

    # ------------------------------------------------------------------
    # S5. Direct Layer Scalar Modulation
    # ------------------------------------------------------------------
    def modulate_layer_scalar(self, layer_scales: Dict[int, float]) -> None:
        with torch.no_grad():
            for l, mult in layer_scales.items():
                name = f"layers.{l}.layer_scalar"
                if name not in self.tensor_map:
                    continue
                self._ensure_backup(name)
                orig = self.backups[name].float()
                t = self.tensor_map[name]
                t.data.copy_((orig * mult).to(device=t.device, dtype=t.dtype))
        self.active_log.append(f"modulate_layer_scalar({layer_scales})")

    # ------------------------------------------------------------------
    # S6. Low-Rank Subspace Gain / Contrastive Projection Surgery
    # ------------------------------------------------------------------
    def graft_residual_subspace_gain(
        self,
        module_weight_name: str,
        basis_vectors: torch.Tensor,
        gain_delta: float,
    ) -> None:
        """
        Given orthonormal basis `V` of shape `[d_out, r]` (`V^T V = I_r`) in residual space,
        transforms output projection `W` (`[d_out, d_in]`) via:
            W_new = (I + gain_delta * V V^T) @ W = W + gain_delta * V @ (V^T @ W)
        For `gain_delta > 0`, amplifies components along `V`; for `gain_delta = -1`, projects out `V`!
        Zero runtime FLOP or parameter overhead.
        """
        if module_weight_name not in self.tensor_map:
            raise KeyError(f"{module_weight_name} not found in tensor_map")
        self._ensure_backup(module_weight_name)
        with torch.no_grad():
            w = self.backups[module_weight_name].float()
            v = basis_vectors.detach().cpu().float()
            if v.ndim == 1:
                v = v.unsqueeze(1)
            # Orthonormalize columns via QR
            q, _ = torch.linalg.qr(v)
            w_new = w + float(gain_delta) * (q @ (q.T @ w))
            t = self.tensor_map[module_weight_name]
            t.data.copy_(w_new.to(device=t.device, dtype=t.dtype))
        self.active_log.append(
            f"graft_residual_subspace_gain(target={module_weight_name}, rank={v.shape[1]}, gain_delta={gain_delta})"
        )


In [ ]:
# @title 5. Write Zero-Gradient Subspace Calibration (`surgeries/subspace_calibration.py`)
%%writefile /content/gemma4_surgery/surgeries/subspace_calibration.py
# Copyright 2026 Keshan Sodimana
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Closed-Form (Zero-Gradient) Static-Shape Subspace Extraction for Gemma 4 Weight Surgeries.

All forward passes use left-padded fixed `[batch_size, pad_len]` shapes matching `StaticGemma4Engine`
so TPU/XLA (`torch_tpu`) performs ZERO extra graph compilations!
"""

from typing import Any, Dict, List, Sequence
import torch
import torch.nn as nn


SURFACE_STOP_STRINGS = [
    "\n",
    "\n\n",
    " ",
    "the",
    " the",
    "The",
    "a",
    " a",
    "is",
    " is",
    "of",
    " of",
    "to",
    " to",
    "and",
    " and",
    "in",
    " in",
    ",",
    ".",
    ":",
    ";",
    "(",
    ")",
]


PROSE_FILLER_STRINGS = [
    "the",
    " the",
    "The",
    "a",
    " a",
    "an",
    " an",
    "of",
    " of",
    "to",
    " to",
    "that",
    " that",
    "it",
    " it",
    "Here",
    "Sure",
    "Let",
    "Well",
    "So",
    "Basically",
]


@torch.no_grad()
def compute_prose_filler_subspace(
    lm_head_weight: torch.Tensor,
    tokenizer: Any,
    rank: int = 8,
) -> torch.Tensor:
    """
    Code-safe analytical extraction of the top-`rank` orthonormal unembedding subspace
    spanned strictly by conversational English prose-filler/article tokens (preserving
    all Python whitespace `\n`/` `, punctuation `:()`, and keyword tokens `in`/`is`/`and`).
    """
    tok_ids = set()
    for s in PROSE_FILLER_STRINGS:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            tok_ids.add(ids[0])
    sorted_ids = sorted(tok_ids)
    rows = lm_head_weight[sorted_ids].detach().float().cpu()
    rows = rows / (rows.norm(dim=-1, keepdim=True) + 1e-8)
    u, _, _ = torch.linalg.svd(rows.T, full_matrices=False)
    r = min(rank, u.shape[1])
    return u[:, :r].contiguous()


@torch.no_grad()
def compute_surface_stop_subspace(
    lm_head_weight: torch.Tensor,
    tokenizer: Any,
    rank: int = 8,
) -> torch.Tensor:
    """
    Analytical (zero-data) extraction of the top-`rank` orthonormal unembedding subspace
    spanned by high-frequency surface/formatting/stop tokens (`\n`, `:`, `the`, etc.).
    """
    tok_ids = set()
    for s in SURFACE_STOP_STRINGS:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            tok_ids.add(ids[0])
    sorted_ids = sorted(tok_ids)
    rows = lm_head_weight[sorted_ids].detach().float().cpu()  # [K, D]
    rows = rows / (rows.norm(dim=-1, keepdim=True) + 1e-8)
    u, _, _ = torch.linalg.svd(rows.T, full_matrices=False)
    r = min(rank, u.shape[1])
    return u[:, :r].contiguous()


def _build_forbidden_tag_basis(lm_head_weight: torch.Tensor, tokenizer: Any) -> torch.Tensor:
    tag_ids = set()
    for tag in ["<|channel>", "<channel|>", "<|think|>", "thought", "<|thought|>"]:
        tid = tokenizer.convert_tokens_to_ids(tag)
        if isinstance(tid, int) and tid >= 0 and tid != tokenizer.unk_token_id:
            tag_ids.add(tid)
        tag_ids.update(tokenizer.encode(tag, add_special_tokens=False))
    tag_vecs = lm_head_weight[sorted(tag_ids)].detach().float().cpu()
    q_forbid, _ = torch.linalg.qr(tag_vecs.T)
    return q_forbid


@torch.no_grad()
def extract_last_token_states_static(
    text_model: nn.Module,
    tokenizer: Any,
    formatted_prompts: Sequence[str],
    target_layers: Sequence[int],
    batch_size: int = 32,
    pad_len: int = 512,
    device: torch.device = torch.device("cpu"),
) -> Dict[int, torch.Tensor]:
    """
    Extract final-prompt-token residual states `[N, hidden_size]` at `target_layers` using
    strictly fixed `[batch_size, pad_len]` left-padded batches (zero XLA recompilation!).
    """
    pad_id = int(getattr(text_model.config, "pad_token_id", 0) or 0)
    sliding_window = int(getattr(text_model.config, "sliding_window", 512) or 512)
    dtype = next(text_model.parameters()).dtype

    captured: Dict[int, torch.Tensor] = {}
    hooks = []

    def make_hook(l_idx: int):
        def _hook(_mod, _inp, out):
            h = out[0] if isinstance(out, tuple) else out
            captured[l_idx] = h[:, -1, :].detach().float().cpu()

        return _hook

    for l in target_layers:
        hooks.append(text_model.layers[l].register_forward_hook(make_hook(l)))

    out_lists: Dict[int, List[torch.Tensor]] = {l: [] for l in target_layers}
    n = len(formatted_prompts)
    zero_bf16 = torch.zeros((), device=device, dtype=dtype)
    neg_inf_bf16 = torch.full((), torch.finfo(dtype).min, device=device, dtype=dtype)
    arange_p = torch.arange(pad_len, device=device, dtype=torch.long)
    q_idx = arange_p[None, None, :, None]
    k_idx = arange_p[None, None, None, :]

    try:
        for start in range(0, n, batch_size):
            chunk = list(formatted_prompts[start : start + batch_size])
            real_n = len(chunk)
            if real_n < batch_size:
                chunk.extend([chunk[-1]] * (batch_size - real_n))

            inp_cpu = torch.full((batch_size, pad_len), pad_id, dtype=torch.long)
            lens_cpu = torch.zeros((batch_size,), dtype=torch.long)
            for i, txt in enumerate(chunk):
                ids = tokenizer.encode(txt, add_special_tokens=False)[-pad_len:]
                L = len(ids)
                inp_cpu[i, pad_len - L : pad_len] = torch.tensor(ids, dtype=torch.long)
                lens_cpu[i] = L

            input_ids = inp_cpu.to(device)
            valid_start = (pad_len - lens_cpu.to(device))[:, None, None, None]
            is_pad_q = q_idx < valid_start
            pad_self = is_pad_q & (k_idx == q_idx)
            real_causal = (~is_pad_q) & (k_idx >= valid_start) & (k_idx <= q_idx)
            masks = {
                "full_attention": torch.where(pad_self | real_causal, zero_bf16, neg_inf_bf16),
                "sliding_attention": torch.where(
                    pad_self | (real_causal & ((q_idx - k_idx) < sliding_window)),
                    zero_bf16,
                    neg_inf_bf16,
                ),
            }
            pos_ids = (q_idx.squeeze(1).squeeze(-1) - valid_start.squeeze(-1).squeeze(-1)).clamp(min=0)

            captured.clear()
            _ = text_model(
                input_ids=input_ids,
                attention_mask=masks,
                position_ids=pos_ids,
                use_cache=False,
            )
            if device.type == "tpu" and hasattr(torch.accelerator, "synchronize"):
                torch.accelerator.synchronize()

            for l in target_layers:
                out_lists[l].append(captured[l][:real_n])
    finally:
        for hk in hooks:
            hk.remove()

    return {l: torch.cat(out_lists[l], dim=0) for l in target_layers}


@torch.no_grad()
def compute_latent_think_subspace(
    text_model: nn.Module,
    lm_head_weight: torch.Tensor,
    tokenizer: Any,
    dev_prompts: Sequence[str],
    target_layers: Sequence[int] = (33, 34),
    rank: int = 4,
    batch_size: int = 32,
    pad_len: int = 512,
    device: torch.device = torch.device("cpu"),
) -> Dict[int, torch.Tensor]:
    """
    Forward-only extraction of top-`rank` principal directions of `h(prompt|think) - h(prompt|direct)`
    on `dev_prompts` in fixed `[batch_size, pad_len]` shape, orthogonalized against thought tags.
    """
    q_forbid = _build_forbidden_tag_basis(lm_head_weight, tokenizer)
    fmt_dir = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        for p in dev_prompts
    ]
    fmt_thk = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True, enable_thinking=True
        )
        for p in dev_prompts
    ]
    states_dir = extract_last_token_states_static(
        text_model, tokenizer, fmt_dir, target_layers, batch_size=batch_size, pad_len=pad_len, device=device
    )
    states_thk = extract_last_token_states_static(
        text_model, tokenizer, fmt_thk, target_layers, batch_size=batch_size, pad_len=pad_len, device=device
    )

    basis_by_layer: Dict[int, torch.Tensor] = {}
    for l in target_layers:
        mat = (states_thk[l] - states_dir[l]).T  # [D, N_dev]
        mat = mat - q_forbid @ (q_forbid.T @ mat)
        u, _, _ = torch.linalg.svd(mat, full_matrices=False)
        r = min(rank, u.shape[1])
        basis_by_layer[l] = u[:, :r].contiguous()
    return basis_by_layer


@torch.no_grad()
def compute_domain_centered_correctness_subspace(
    text_model: nn.Module,
    lm_head_weight: torch.Tensor,
    tokenizer: Any,
    dev_suites_prompts: Dict[str, Sequence[str]],
    dev_baseline_correct: Dict[str, Sequence[bool]],
    target_layers: Sequence[int] = (33, 34),
    rank: int = 2,
    batch_size: int = 32,
    pad_len: int = 512,
    device: torch.device = torch.device("cpu"),
) -> Dict[int, torch.Tensor]:
    """
    Extract domain-invariant contrastive correctness subspace across `dev` benchmarks:
    Within each benchmark `b`, computes unit-normalized `mean(h | correct, b) - mean(h | wrong, b)`
    so topic differences across benchmarks cancel out, then takes top-`rank` SVD across domains!
    """
    stop_q = compute_surface_stop_subspace(lm_head_weight, tokenizer, rank=8)
    domain_diffs: Dict[int, List[torch.Tensor]] = {l: [] for l in target_layers}

    for bname, raw_prompts in dev_suites_prompts.items():
        flags = list(dev_baseline_correct[bname])
        if sum(flags) == 0 or sum(flags) == len(flags):
            continue
        fmt = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            for p in raw_prompts
        ]
        st = extract_last_token_states_static(
            text_model, tokenizer, fmt, target_layers, batch_size=batch_size, pad_len=pad_len, device=device
        )
        pos_idx = torch.tensor([i for i, f in enumerate(flags) if f], dtype=torch.long)
        neg_idx = torch.tensor([i for i, f in enumerate(flags) if not f], dtype=torch.long)
        for l in target_layers:
            diff = st[l][pos_idx].mean(dim=0) - st[l][neg_idx].mean(dim=0)
            diff = diff - stop_q @ (stop_q.T @ diff)
            diff = diff / (diff.norm() + 1e-8)
            domain_diffs[l].append(diff)

    basis_by_layer: Dict[int, torch.Tensor] = {}
    for l in target_layers:
        mat = torch.stack(domain_diffs[l], dim=1)  # [D, num_domains]
        u, _, _ = torch.linalg.svd(mat, full_matrices=False)
        r = min(rank, u.shape[1])
        basis_by_layer[l] = u[:, :r].contiguous()
    return basis_by_layer


In [ ]:
# @title 6. Run Phase 1 35-Layer Spectral & Activation MRI Probe
# Copyright 2026 Keshan Sodimana
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Phase 1: Mechanistic & Spectral Anatomy Probe for Gemma 4 E2B-it on Google Colab TPU v5e-1 / GPU.

Persists `MODEL`, `TOKENIZER`, `TEXT_MODEL`, `DEVICE`, `REPO_ID` in live Colab kernel globals
so subsequent weight-surgery and benchmark scripts execute without reloading weights.
"""

import inspect
import json
import math
import os
import time
from pathlib import Path

import torch
import transformers


def get_device() -> torch.device:
    if hasattr(torch, "accelerator") and torch.accelerator.is_available():
        return torch.accelerator.current_accelerator()
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def sync_device() -> None:
    if hasattr(torch, "accelerator") and torch.accelerator.is_available():
        torch.accelerator.synchronize()
    elif torch.cuda.is_available():
        torch.cuda.synchronize()


def get_hardware_info(device: torch.device) -> dict:
    info = {
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "device": str(device),
        "cpu_count": os.cpu_count(),
    }
    if device.type == "cuda":
        info["device_name"] = torch.cuda.get_device_name(0)
        info["vram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    elif device.type == "tpu":
        info["device_name"] = "Google Cloud TPU v5e-1 (Colab Free)"
        info["hbm_gb"] = 15.75
    return info


def stable_rank_and_spectrum(weight: torch.Tensor) -> dict:
    """Compute SVD spectrum statistics on CPU float32 (avoids XLA shape recompilations)."""
    w = weight.detach().to(device="cpu", dtype=torch.float32)
    if w.ndim > 2:
        w = w.reshape(w.shape[0], -1)
    s = torch.linalg.svdvals(w)
    top1 = float(s[0].item())
    frob_sq = float((s**2).sum().item())
    srank = frob_sq / (top1**2 + 1e-12)
    p = (s**2) / (frob_sq + 1e-12)
    p_pos = p[p > 1e-12]
    eff_rank = float(torch.exp(-(p_pos * torch.log(p_pos)).sum()).item())
    return {
        "shape": list(weight.shape),
        "top1_sv": round(top1, 5),
        "sv_median": round(float(s[len(s) // 2].item()), 5),
        "sv_min": round(float(s[-1].item()), 6),
        "frob_norm": round(math.sqrt(frob_sq), 5),
        "stable_rank": round(srank, 3),
        "effective_rank": round(eff_rank, 3),
        "top10_energy_ratio": round(float((s[:10] ** 2).sum().item()) / (frob_sq + 1e-12), 5),
        "top64_energy_ratio": round(float((s[:64] ** 2).sum().item()) / (frob_sq + 1e-12), 5),
    }


def load_or_reuse_gemma4(device: torch.device, force_reload: bool = False):
    g = globals()
    if (not force_reload) and "MODEL" in g and g["MODEL"] is not None and type(g["MODEL"]).__name__ == "Gemma4ForConditionalGeneration":
        print(f"[probe] Reusing already-loaded {g.get('REPO_ID')} on {device}")
        return g["REPO_ID"], g["CFG"], g["TOKENIZER"], g["MODEL"]

    # Free any wrongly-loaded causal wrapper first
    if "MODEL" in g and g["MODEL"] is not None:
        del g["MODEL"]
        import gc
        gc.collect()

    from transformers import AutoConfig, AutoTokenizer, Gemma4ForConditionalGeneration

    candidates = ["unsloth/gemma-4-E2B-it", "google/gemma-4-E2B-it"]
    hf_token = os.environ.get("HF_TOKEN") or None
    if hf_token:
        candidates = ["google/gemma-4-E2B-it", "unsloth/gemma-4-E2B-it"]

    last_err = None
    for repo_id in candidates:
        try:
            print(f"[probe] Loading {repo_id} via Gemma4ForConditionalGeneration (bfloat16)...", flush=True)
            t0 = time.time()
            cfg = AutoConfig.from_pretrained(repo_id, token=hf_token)
            tokenizer = AutoTokenizer.from_pretrained(repo_id, token=hf_token)
            model = Gemma4ForConditionalGeneration.from_pretrained(
                repo_id,
                dtype=torch.bfloat16,
                token=hf_token,
            )
            # Offload unused multimodal encoders from TPU HBM while keeping 100% of language_model + lm_head intact
            inner = getattr(model, "model", model)
            for tower_name in ["vision_tower", "audio_tower", "embed_vision", "embed_audio"]:
                if hasattr(inner, tower_name) and getattr(inner, tower_name) is not None:
                    setattr(inner, tower_name, None)
            import gc
            gc.collect()

            model.eval()
            model.to(device)
            sync_device()
            print(f"[probe] Gemma4ForConditionalGeneration ready on {device} in {time.time() - t0:.2f}s", flush=True)
            return repo_id, cfg, tokenizer, model
        except Exception as e:
            print(f"[probe] Candidate {repo_id} failed: {e}", flush=True)
            last_err = e
    raise RuntimeError(f"Could not load Gemma 4 E2B-it: {last_err}")


def locate_text_backbone(model):
    for attr_path in [
        ("model", "language_model"),
        ("language_model", "model"),
        ("language_model",),
        ("text_model",),
        ("model",),
    ]:
        obj = model
        ok = True
        for a in attr_path:
            if hasattr(obj, a):
                obj = getattr(obj, a)
            else:
                ok = False
                break
        if ok and hasattr(obj, "layers"):
            return ".".join(attr_path), obj
    raise AttributeError(f"Cannot locate decoder layers in {type(model).__name__}: {[n for n, _ in model.named_children()]}")


device = get_device()
out_dir = Path("/content/gemma4_probe")
out_dir.mkdir(parents=True, exist_ok=True)

report = {"hardware": get_hardware_info(device)}
print("[probe] Hardware:", json.dumps(report["hardware"], indent=2), flush=True)

REPO_ID, CFG, TOKENIZER, MODEL = load_or_reuse_gemma4(device)
DEVICE = device
BACKBONE_PATH, TEXT_MODEL = locate_text_backbone(MODEL)

report["repo_id"] = REPO_ID
report["model_class"] = type(MODEL).__name__
report["backbone_path"] = BACKBONE_PATH
report["text_model_class"] = type(TEXT_MODEL).__name__
report["decoder_layer_class"] = type(TEXT_MODEL.layers[0]).__name__

# Dump exact modeling_gemma4 module source file
import transformers.models.gemma4.modeling_gemma4 as mod_g4

mod_src = inspect.getsource(mod_g4)
(out_dir / "modeling_gemma4_exact.py").write_text(mod_src)
print(f"[probe] Dumped modeling_gemma4.py ({len(mod_src)} chars)", flush=True)

# Parameter census
total_params = sum(p.numel() for p in MODEL.parameters())
text_params = sum(p.numel() for p in TEXT_MODEL.parameters())
ple_table = getattr(TEXT_MODEL, "embed_tokens_per_layer", None)
ple_table_params = ple_table.weight.numel() if ple_table is not None else 0
embed_params = TEXT_MODEL.embed_tokens.weight.numel() if hasattr(TEXT_MODEL, "embed_tokens") else 0

report["param_census"] = {
    "total_loaded_params": total_params,
    "text_backbone_params": text_params,
    "main_embed_tokens_params": embed_params,
    "ple_embed_table_params": ple_table_params,
    "non_embedding_compute_params": text_params - embed_params - ple_table_params,
    "num_layers": len(TEXT_MODEL.layers),
    "text_config": TEXT_MODEL.config.to_dict(),
    "text_model_children": [name for name, _ in TEXT_MODEL.named_children()],
    "layer0_children": [name for name, _ in TEXT_MODEL.layers[0].named_children()],
    "attn0_children": [name for name, _ in TEXT_MODEL.layers[0].self_attn.named_children()],
    "layer_last_attn_children": [name for name, _ in TEXT_MODEL.layers[-1].self_attn.named_children()],
}
print("[probe] Census:", json.dumps({k: v for k, v in report["param_census"].items() if k != "text_config"}, indent=2), flush=True)

# Layer-by-layer weight & spectral anatomy
print("[probe] Computing SVD & norm anatomy across all layers...", flush=True)
layer_stats = []
for idx, layer in enumerate(TEXT_MODEL.layers):
    entry = {"layer": idx}
    for attr in ["layer_type", "is_sliding"]:
        if hasattr(layer, attr):
            entry[attr] = getattr(layer, attr)
    attn = layer.self_attn
    for attr in ["is_kv_shared_layer", "kv_shared_layer_index", "is_sliding", "head_dim", "use_k_eq_v", "k_eq_v"]:
        if hasattr(attn, attr):
            val = getattr(attn, attr)
            entry[f"attn_{attr}"] = val if isinstance(val, (int, float, bool, str, type(None))) else str(val)
    entry["attn_submodules"] = [n for n, _ in attn.named_children()]

    if hasattr(layer, "layer_scalar") and layer.layer_scalar is not None:
        ls = layer.layer_scalar.detach().to("cpu", dtype=torch.float32)
        entry["layer_scalar"] = round(float(ls.mean().item()), 6) if ls.numel() == 1 else [round(x, 5) for x in ls.flatten()[:8].tolist()]

    for norm_name in [
        "input_layernorm",
        "post_attention_layernorm",
        "pre_feedforward_layernorm",
        "post_feedforward_layernorm",
        "post_per_layer_input_norm",
    ]:
        mod_n = getattr(layer, norm_name, None)
        if mod_n is not None and hasattr(mod_n, "weight") and mod_n.weight is not None:
            nw = mod_n.weight.detach().to("cpu", dtype=torch.float32)
            entry[f"{norm_name}_rms"] = round(float(nw.pow(2).mean().sqrt().item()), 5)
            entry[f"{norm_name}_mean"] = round(float(nw.mean().item()), 5)

    for qk_norm in ["q_norm", "k_norm", "v_norm"]:
        mod_qk = getattr(attn, qk_norm, None)
        if mod_qk is not None and hasattr(mod_qk, "weight") and mod_qk.weight is not None:
            nw = mod_qk.weight.detach().to("cpu", dtype=torch.float32)
            entry[f"attn_{qk_norm}_rms"] = round(float(nw.pow(2).mean().sqrt().item()), 5)

    if hasattr(layer, "per_layer_input_gate") and layer.per_layer_input_gate is not None:
        entry["ple_gate_svd"] = stable_rank_and_spectrum(layer.per_layer_input_gate.weight)
    if hasattr(layer, "per_layer_projection") and layer.per_layer_projection is not None:
        entry["ple_proj_svd"] = stable_rank_and_spectrum(layer.per_layer_projection.weight)
    if hasattr(attn, "q_proj") and attn.q_proj is not None:
        entry["q_proj_svd"] = stable_rank_and_spectrum(attn.q_proj.weight)
    if hasattr(attn, "o_proj") and attn.o_proj is not None:
        entry["o_proj_svd"] = stable_rank_and_spectrum(attn.o_proj.weight)
    if hasattr(layer.mlp, "down_proj") and layer.mlp.down_proj is not None:
        entry["mlp_down_svd"] = stable_rank_and_spectrum(layer.mlp.down_proj.weight)

    layer_stats.append(entry)

report["layer_weight_anatomy"] = layer_stats

# Inspect PLE embedding table norms across token types
if ple_table is not None:
    ple_w = ple_table.weight.detach().to("cpu", dtype=torch.float32)  # [V, L * d_ple]
    n_layers = len(TEXT_MODEL.layers)
    d_ple = ple_w.shape[1] // n_layers
    ple_w_3d = ple_w.view(ple_w.shape[0], n_layers, d_ple)
    sample_tokens = {
        "syntax_stop": ["the", " is", ",", ".", "\n", " of", " and", "="],
        "numbers_math": ["2", "16", "100", "+", "*", "/", "x", "0"],
        "domain_content": ["quantum", "mitochondria", "eigenvalue", "Python", "derivative", "velocity", "theorem", "algorithm"],
    }
    ple_token_probe = {}
    for group, toks in sample_tokens.items():
        ids = []
        for t in toks:
            enc_ids = TOKENIZER.encode(t, add_special_tokens=False)
            if enc_ids:
                ids.append(enc_ids[-1])
        sub = ple_w_3d[ids]  # [k, L, d_ple]
        norms_by_layer = sub.norm(dim=-1).mean(dim=0).tolist()
        ple_token_probe[group] = {
            "token_ids": ids,
            "mean_norm_L0": round(norms_by_layer[0], 5),
            "mean_norm_L14": round(norms_by_layer[14], 5),
            "mean_norm_L25": round(norms_by_layer[25], 5),
            "mean_norm_L34": round(norms_by_layer[-1], 5),
            "all_layers_mean_norm": [round(x, 4) for x in norms_by_layer],
        }
    report["ple_token_group_norms"] = ple_token_probe

# Check special tokens and chat template (including thinking mode!)
special_probe = {}
for tok_str in ["<|think|>", "<|thought|>", "<thought>", "</thought>", "<start_of_turn>", "<end_of_turn>"]:
    special_probe[tok_str] = TOKENIZER.convert_tokens_to_ids(tok_str)
report["special_tokens"] = special_probe
report["chat_template"] = getattr(TOKENIZER, "chat_template", None)

# Activation MRI on diagnostic reasoning prompts
prompts = [
    "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
    "In quantum mechanics, consider a particle in a 1D infinite square well of width L. If the width is suddenly doubled to 2L without changing the wavefunction, what is the probability of finding the particle in the new ground state?",
    "Write a Python function `first_repeated_char(s)` to find the first repeated character in a given string.",
]

formatted = []
for p in prompts:
    msgs = [{"role": "user", "content": p}]
    try:
        formatted.append(TOKENIZER.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    except Exception:
        formatted.append(p)
report["formatted_prompt_sample"] = formatted[0]

act_records = {
    i: {
        "h_in_norm": [],
        "attn_post_norm": [],
        "mlp_post_norm": [],
        "ple_pre_norm_mean": [],
        "ple_pre_norm_cv": [],
        "ple_post_norm_mean": [],
        "ple_post_norm_cv": [],
        "h_out_norm": [],
        "cos_in_out": [],
    }
    for i in range(len(TEXT_MODEL.layers))
}

hooks = []
for idx, layer in enumerate(TEXT_MODEL.layers):
    if getattr(layer, "post_attention_layernorm", None) is not None:
        hooks.append(
            layer.post_attention_layernorm.register_forward_hook(
                lambda _m, _i, o, l=idx: act_records[l]["attn_post_norm"].append(
                    float(o.detach().to("cpu", dtype=torch.float32).norm(dim=-1).mean().item())
                )
            )
        )
    if getattr(layer, "post_feedforward_layernorm", None) is not None:
        hooks.append(
            layer.post_feedforward_layernorm.register_forward_hook(
                lambda _m, _i, o, l=idx: act_records[l]["mlp_post_norm"].append(
                    float(o.detach().to("cpu", dtype=torch.float32).norm(dim=-1).mean().item())
                )
            )
        )
    if getattr(layer, "per_layer_projection", None) is not None:
        def _mk_pre(l):
            def _hk(_m, _i, o):
                n = o.detach().to("cpu", dtype=torch.float32).norm(dim=-1)
                act_records[l]["ple_pre_norm_mean"].append(float(n.mean().item()))
                act_records[l]["ple_pre_norm_cv"].append(float((n.std() / (n.mean() + 1e-8)).item()))
            return _hk
        hooks.append(layer.per_layer_projection.register_forward_hook(_mk_pre(idx)))
    if getattr(layer, "post_per_layer_input_norm", None) is not None:
        def _mk_post(l):
            def _hk(_m, _i, o):
                n = o.detach().to("cpu", dtype=torch.float32).norm(dim=-1)
                act_records[l]["ple_post_norm_mean"].append(float(n.mean().item()))
                act_records[l]["ple_post_norm_cv"].append(float((n.std() / (n.mean() + 1e-8)).item()))
            return _hk
        hooks.append(layer.post_per_layer_input_norm.register_forward_hook(_mk_post(idx)))

    def _mk_layer(l):
        def _hk(_m, inp, out):
            hi = inp[0].detach().to("cpu", dtype=torch.float32)
            ho = (out[0] if isinstance(out, tuple) else out).detach().to("cpu", dtype=torch.float32)
            act_records[l]["h_in_norm"].append(float(hi.norm(dim=-1).mean().item()))
            act_records[l]["h_out_norm"].append(float(ho.norm(dim=-1).mean().item()))
            cos = torch.nn.functional.cosine_similarity(hi, ho, dim=-1)
            act_records[l]["cos_in_out"].append(float(cos.mean().item()))
        return _hk
    hooks.append(layer.register_forward_hook(_mk_layer(idx)))

print("[probe] Running forward activation MRI...", flush=True)
with torch.no_grad():
    for text in formatted:
        enc = TOKENIZER(text, return_tensors="pt").to(device)
        _ = MODEL(**enc)
        sync_device()

for h in hooks:
    h.remove()

def _avg(lst):
    return round(sum(lst) / max(1, len(lst)), 5)

act_summary = []
for idx in range(len(TEXT_MODEL.layers)):
    r = act_records[idx]
    act_summary.append({
        "layer": idx,
        "h_in_norm": _avg(r["h_in_norm"]),
        "attn_update_norm": _avg(r["attn_post_norm"]),
        "mlp_update_norm": _avg(r["mlp_post_norm"]),
        "ple_pre_norm_mean": _avg(r["ple_pre_norm_mean"]),
        "ple_pre_norm_cv": _avg(r["ple_pre_norm_cv"]),
        "ple_post_norm_mean": _avg(r["ple_post_norm_mean"]),
        "ple_post_norm_cv": _avg(r["ple_post_norm_cv"]),
        "h_out_norm": _avg(r["h_out_norm"]),
        "cos_in_out": _avg(r["cos_in_out"]),
    })
report["activation_mri"] = act_summary

# Quick generation & latency smoke test (warmup + timed decode)
print("[probe] Running generation & throughput smoke test...", flush=True)
test_enc = TOKENIZER(formatted[0], return_tensors="pt").to(device)
with torch.no_grad():
    _ = MODEL.generate(**test_enc, max_new_tokens=16, do_sample=False)
    sync_device()
    t_gen = time.time()
    gen_out = MODEL.generate(**test_enc, max_new_tokens=96, do_sample=False)
    sync_device()
    dt_gen = time.time() - t_gen

new_toks = gen_out.shape[1] - test_enc["input_ids"].shape[1]
gen_text = TOKENIZER.decode(gen_out[0, test_enc["input_ids"].shape[1]:], skip_special_tokens=False)
report["smoke_generation"] = {
    "new_tokens": int(new_toks),
    "elapsed_sec": round(dt_gen, 3),
    "tokens_per_sec": round(new_toks / max(dt_gen, 1e-6), 2),
    "output_preview": gen_text,
}
print(f"[probe] Smoke decode: {new_toks} toks in {dt_gen:.2f}s ({new_toks/max(dt_gen,1e-6):.1f} tok/s)", flush=True)
print("[probe] Output preview:", gen_text[:300], flush=True)

out_json = out_dir / "phase1_anatomy_report.json"
out_json.write_text(json.dumps(report, indent=2))
print(f"\n[probe] Saved complete MRI report -> {out_json}", flush=True)

print("\n=== LAYER-BY-LAYER MRI SUMMARY ===")
for row in act_summary:
    l = row["layer"]
    w = layer_stats[l]
    print(
        f"L{l:02d} ({str(w.get('layer_type','?'))[:7]:7s}, kv_sh={int(bool(w.get('attn_is_kv_shared_layer')))}<-{str(w.get('attn_kv_shared_layer_index')):>4s}) | "
        f"sc={w.get('layer_scalar', 1.0):.3f} | "
        f"h:{row['h_in_norm']:6.1f}->{row['h_out_norm']:6.1f} | "
        f"attn={row['attn_update_norm']:5.1f} mlp={row['mlp_update_norm']:5.1f} ple={row['ple_post_norm_mean']:5.1f} | "
        f"PLE_CV:{row['ple_pre_norm_cv']:.3f}->{row['ple_post_norm_cv']:.3f} | "
        f"cos={row['cos_in_out']:.4f}"
    )


In [ ]:
# @title 7. Run Held-Out Test Evaluation (Baseline vs. Surgical Models)
# Copyright 2026 Keshan Sodimana
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Exp 4: Modular Checkpointed Held-Out `test`-Split Evaluation (`N = 384`: 96 per benchmark).

Supports incremental execution via `os.environ.get("EXP4_STAGE", "all")`:
  - `EXP4_STAGE=baseline` -> loads model/test suites, runs `00_baseline`, caches `EXP4_STATE` in kernel & JSON
  - `EXP4_STAGE=S1d`      -> runs `S1d_prose_orth_proj_25_34` against cached baseline & updates JSON
  - `EXP4_STAGE=S1c`      -> runs `S1c_ple_stop_subspace_clean` against cached baseline & updates JSON
  - `EXP4_STAGE=S6b`      -> runs `S6b_bypass_L31_comp_plus_prose_orth` against cached baseline & updates JSON
  - `EXP4_STAGE=all`      -> runs baseline + all three locked finalists sequentially
"""

import json
import os
import subprocess
import sys
import tarfile
from typing import Any, Callable, Dict, List

try:
    import datasets  # noqa: F401
    from transformers import Gemma4ForConditionalGeneration  # noqa: F401
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "datasets", "accelerate", "safetensors"]
    )

import torch

if os.path.exists("/content/gemma4_pkg.tar.gz"):
    with tarfile.open("/content/gemma4_pkg.tar.gz", "r:gz") as tf:
        tf.extractall("/content", filter="data")

for _mod in list(sys.modules):
    if _mod == "benchmarks" or _mod.startswith("benchmarks.") or _mod == "surgeries" or _mod.startswith("surgeries."):
        del sys.modules[_mod]

sys.path.insert(0, "/content/gemma4_surgery")
from benchmarks.eval_harness import (
    grade_item,
    load_arc_challenge,
    load_gsm8k,
    load_mbpp_sanitized,
    load_mmlu_pro_stem,
    paired_significance,
)
from benchmarks.fast_infer import StaticGemma4Engine
from surgeries.subspace_calibration import (
    compute_prose_filler_subspace,
    compute_surface_stop_subspace,
)
from surgeries.weight_surgeon import WeightSurgeon

OUT_PATH = "/content/gemma4_surgery/results/exp4_heldout_test_results.json"
os.makedirs("/content/gemma4_surgery/results", exist_ok=True)

STAGE = os.environ.get("EXP4_STAGE", "all").strip()

if "MODEL" not in globals():
    from transformers import AutoTokenizer, Gemma4ForConditionalGeneration

    DEVICE = torch.accelerator.current_accelerator() if hasattr(torch, "accelerator") else torch.device("cpu")
    REPO_ID = "unsloth/gemma-4-E2B-it"
    print(f"[Loading {REPO_ID} onto {DEVICE}...]", flush=True)
    TOKENIZER = AutoTokenizer.from_pretrained(REPO_ID)
    MODEL = Gemma4ForConditionalGeneration.from_pretrained(REPO_ID, dtype=torch.bfloat16)
    MODEL.model.vision_tower = None
    MODEL.model.audio_tower = None
    MODEL.model.embed_vision = None
    MODEL.model.embed_audio = None
    MODEL = MODEL.to(DEVICE)
    MODEL.eval()

if "EXP4_ENGINE" not in globals():
    EXP4_ENGINE = StaticGemma4Engine(MODEL, TOKENIZER, DEVICE)
if "EXP4_SURGEON" not in globals():
    EXP4_SURGEON = WeightSurgeon(EXP4_ENGINE.text_model)

BATCH_SIZE = 32
PAD_BUCKET = 512
MAX_NEW_TOKENS = 257

if "EXP4_TEST_SUITES" not in globals():
    EXP4_TEST_SUITES = {
        "gsm8k": load_gsm8k(split_mode="test", n=96, seed=2026),
        "arc_challenge": load_arc_challenge(split_mode="test", n=96, seed=2026),
        "mmlu_pro": load_mmlu_pro_stem(split_mode="test", n=96, seed=2026),
        "mbpp": load_mbpp_sanitized(split_mode="test", n=96, seed=2026),
    }
    print("Loaded HELD-OUT TEST suites:", {k: len(v) for k, v in EXP4_TEST_SUITES.items()}, flush=True)

PROSE_BASIS = compute_prose_filler_subspace(EXP4_ENGINE.lm_head.weight, TOKENIZER, rank=8)
STOP_BASIS = compute_surface_stop_subspace(EXP4_ENGINE.lm_head.weight, TOKENIZER, rank=8)

LOCKED_RECIPES: Dict[str, tuple[str, Callable[[WeightSurgeon], None]]] = {
    "S1d": (
        "S1d_prose_orth_proj_25_34",
        lambda s: [
            s.graft_residual_subspace_gain(f"layers.{l}.per_layer_projection.weight", PROSE_BASIS, -1.0)
            for l in range(25, 35)
        ],
    ),
    "S1c": (
        "S1c_ple_stop_subspace_clean",
        lambda s: [
            s.graft_residual_subspace_gain(f"layers.{l}.per_layer_projection.weight", STOP_BASIS, -0.80)
            for l in range(25, 35)
        ],
    ),
    "S6b": (
        "S6b_bypass_L31_comp_plus_prose_orth",
        lambda s: (
            [
                s.graft_residual_subspace_gain(f"layers.{l}.per_layer_projection.weight", PROSE_BASIS, -1.0)
                for l in range(25, 35)
            ],
            s.bypass_kv_shared_layers([31], compensate_scalar=True),
        ),
    ),
}


def run_test_eval(
    tag: str,
    baseline_flags: Dict[str, List[bool]] | None = None,
    baseline_texts: Dict[str, List[str]] | None = None,
) -> tuple[Dict[str, Any], Dict[str, List[str]]]:
    suite_results: Dict[str, Any] = {}
    raw_texts: Dict[str, List[str]] = {}
    qualitative_wins: List[Dict[str, Any]] = []
    qualitative_losses: List[Dict[str, Any]] = []

    total_correct = 0
    total_items = 0
    total_tokens = 0
    total_sec = 0.0

    for bname, items in EXP4_TEST_SUITES.items():
        prompts = [it["prompt"] for it in items]
        gen = EXP4_ENGINE.generate_dataset(
            prompts,
            batch_size=BATCH_SIZE,
            max_new_tokens=MAX_NEW_TOKENS,
            prompt_pad_bucket=PAD_BUCKET,
            enable_thinking=False,
            sync_every=4,
        )
        raw_texts[bname] = gen["texts"]
        correct_flags: List[bool] = []
        for idx, (it, out_txt) in enumerate(zip(items, gen["texts"])):
            ok, _ = grade_item(it, out_txt)
            correct_flags.append(bool(ok))

            if baseline_flags is not None and baseline_texts is not None:
                b_ok = baseline_flags[bname][idx]
                if (not b_ok) and ok and len(qualitative_wins) < 20:
                    qualitative_wins.append({
                        "id": it["id"],
                        "benchmark": bname,
                        "prompt": it["prompt"][:500],
                        "target": str(it["target"])[:200],
                        "baseline_output": baseline_texts[bname][idx][:600],
                        "surgery_output": out_txt[:600],
                    })
                elif b_ok and (not ok) and len(qualitative_losses) < 10:
                    qualitative_losses.append({
                        "id": it["id"],
                        "benchmark": bname,
                        "prompt": it["prompt"][:500],
                        "target": str(it["target"])[:200],
                        "baseline_output": baseline_texts[bname][idx][:600],
                        "surgery_output": out_txt[:600],
                    })

        acc = sum(correct_flags) / len(correct_flags)
        entry: Dict[str, Any] = {
            "n": len(items),
            "acc": round(acc, 4),
            "num_correct": sum(correct_flags),
            "total_tokens": gen["total_tokens"],
            "elapsed_sec": gen["total_elapsed_sec"],
            "tok_per_sec": gen["tok_per_sec"],
            "correct_flags": correct_flags,
        }
        if baseline_flags is not None:
            entry["vs_baseline"] = paired_significance(baseline_flags[bname], correct_flags)
        suite_results[bname] = entry

        total_correct += sum(correct_flags)
        total_items += len(items)
        total_tokens += gen["total_tokens"]
        total_sec += gen["total_elapsed_sec"]

    reasoning_keys = ["gsm8k", "arc_challenge", "mmlu_pro"]
    reas_corr = sum(suite_results[k]["num_correct"] for k in reasoning_keys)
    reas_n = sum(suite_results[k]["n"] for k in reasoning_keys)

    res: Dict[str, Any] = {
        "tag": tag,
        "active_surgeries": list(EXP4_SURGEON.active_log),
        "overall_acc": round(total_correct / total_items, 4),
        "total_correct": total_correct,
        "total_items": total_items,
        "reasoning_3bench_acc": round(reas_corr / reas_n, 4),
        "reasoning_3bench_correct": reas_corr,
        "reasoning_3bench_n": reas_n,
        "overall_tok_per_sec": round(total_tokens / max(total_sec, 1e-6), 2),
        "total_elapsed_sec": round(total_sec, 2),
        "benchmarks": suite_results,
    }
    if baseline_flags is not None:
        flat_base = [f for b in EXP4_TEST_SUITES for f in baseline_flags[b]]
        flat_surg = [f for b in EXP4_TEST_SUITES for f in suite_results[b]["correct_flags"]]
        res["overall_vs_baseline"] = paired_significance(flat_base, flat_surg)

        reas_base = [f for b in reasoning_keys for f in baseline_flags[b]]
        reas_surg = [f for b in reasoning_keys for f in suite_results[b]["correct_flags"]]
        res["reasoning_3bench_vs_baseline"] = paired_significance(reas_base, reas_surg)
        res["qualitative_wins"] = qualitative_wins
        res["qualitative_losses"] = qualitative_losses

    print(
        f"[TEST | {tag}] overall={res['overall_acc']*100:.2f}% ({total_correct}/{total_items}) | "
        f"reasoning={res['reasoning_3bench_acc']*100:.2f}% ({reas_corr}/{reas_n}) | "
        f"speed={res['overall_tok_per_sec']} tok/s ({res['total_elapsed_sec']}s) | "
        + " | ".join(f"{b}={suite_results[b]['acc']*100:.2f}%" for b in EXP4_TEST_SUITES),
        flush=True,
    )
    return res, raw_texts


if "EXP4_RESULTS" not in globals():
    if os.path.exists(OUT_PATH):
        EXP4_RESULTS = json.load(open(OUT_PATH))
    else:
        EXP4_RESULTS = {"models": {}}

if STAGE in ("baseline", "all"):
    EXP4_SURGEON.restore_all()
    base_res, EXP4_BASE_TEXTS = run_test_eval("00_baseline")
    EXP4_RESULTS["baseline"] = base_res
    EXP4_BASE_FLAGS = {b: base_res["benchmarks"][b]["correct_flags"] for b in EXP4_TEST_SUITES}
    with open(OUT_PATH, "w") as fh:
        json.dump(EXP4_RESULTS, fh, indent=2)

stages_to_run = ["S1d", "S1c", "S6b"] if STAGE == "all" else ([STAGE] if STAGE in LOCKED_RECIPES else [])
for skey in stages_to_run:
    full_name, recipe_fn = LOCKED_RECIPES[skey]
    EXP4_SURGEON.restore_all()
    recipe_fn(EXP4_SURGEON)
    m_res, _ = run_test_eval(full_name, baseline_flags=EXP4_BASE_FLAGS, baseline_texts=EXP4_BASE_TEXTS)
    EXP4_RESULTS["models"][full_name] = m_res
    EXP4_SURGEON.restore_all()
    with open(OUT_PATH, "w") as fh:
        json.dump(EXP4_RESULTS, fh, indent=2)

print(f"Finished EXP4_STAGE={STAGE}! Saved to {OUT_PATH}", flush=True)


In [ ]:
# @title 8. Write Pure JAX Static-Graph Engine (`benchmarks/jax_fast_infer.py`)
%%writefile /content/gemma4_surgery/benchmarks/jax_fast_infer.py
# Copyright 2026 Keshan Sodimana
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Pure JAX / XLA Static-Graph Batched Inference Engine for Gemma 4 E2B-it (`JaxStaticGemma4Engine`).

Implements two idiomatic JAX/XLA static-graph execution modes to compare against PyTorch `StaticGemma4Engine`:
1. `generate_batch_jax_step_loop`:
   Step-by-step `@jax.jit(donate_argnums=...)` loop with `jax.lax.dynamic_update_slice` on the 15-layer
   KV cache (`L0..L14`, `-57.14%` KV HBM) — the direct 1-to-1 JAX analogue to PyTorch `index_copy_`.
2. `generate_batch_jax_scan`:
   Whole-loop compiled XLA execution fusing Prefill + all `G - 1` Autoregressive Decode steps inside a
   single `@jax.jit` + `jax.lax.scan` program, with Zone 1 (`L0..L14`) and Zone 2 (`L15..L34`) structured
   as stacked superblocks. Eliminates 100% of per-token Python host-loop and PJRT dispatch overhead.
"""

import time
from functools import partial
from typing import Any, Dict, List, Optional, Sequence, Tuple

import jax
import jax.numpy as jnp
import numpy as np
import torch


def torch_bf16_to_jax(t: torch.Tensor, device: Any) -> jax.Array:
    """Transfer a PyTorch tensor to a JAX device preserving exact bfloat16/float32 bits."""
    t_cpu = t.detach().cpu().contiguous()
    if t_cpu.dtype == torch.bfloat16:
        arr_u16 = t_cpu.view(torch.uint16).numpy()
        arr_bf16 = arr_u16.view(jnp.bfloat16)
        return jax.device_put(arr_bf16, device)
    elif t_cpu.dtype == torch.float32:
        return jax.device_put(t_cpu.numpy().astype(jnp.float32), device)
    else:
        return jax.device_put(t_cpu.float().numpy().astype(jnp.bfloat16), device)


def rms_norm(x: jax.Array, weight: Optional[jax.Array], eps: float = 1e-6) -> jax.Array:
    """Exact Gemma4RMSNorm in JAX: float32 variance with power(-0.5) matching modeling_gemma4.py."""
    x_f32 = x.astype(jnp.float32)
    mean_sq = jnp.mean(jnp.square(x_f32), axis=-1, keepdims=True) + eps
    normed = x_f32 * jnp.power(mean_sq, -0.5)
    if weight is not None:
        normed = normed * weight.astype(jnp.float32)
    return normed.astype(x.dtype)


def rotate_half(x: jax.Array) -> jax.Array:
    half = x.shape[-1] // 2
    return jnp.concatenate([-x[..., half:], x[..., :half]], axis=-1)


def apply_rope(x: jax.Array, cos: jax.Array, sin: jax.Array) -> jax.Array:
    """`x`: `[B, S, H, D]`, `cos`/`sin`: `[B, S, D]`. Broadcasts over heads `H` (axis 2)."""
    cos_b = jnp.expand_dims(cos, axis=2)
    sin_b = jnp.expand_dims(sin, axis=2)
    return (x * cos_b) + (rotate_half(x) * sin_b)


def repeat_kv_jax(x: jax.Array, n_rep: int) -> jax.Array:
    """`x`: `[B, H_kv, S, D]` -> `[B, H_kv * n_rep, S, D]`."""
    if n_rep == 1:
        return x
    B, H_kv, S, D = x.shape
    return jnp.broadcast_to(x[:, :, None, :, :], (B, H_kv, n_rep, S, D)).reshape(B, H_kv * n_rep, S, D)


def attention_core(
    q: jax.Array,
    k: jax.Array,
    v: jax.Array,
    mask: jax.Array,
    n_rep: int,
) -> jax.Array:
    """
    Exact Gemma4TextAttention core (`scaling = 1.0`, float32 softmax, `bfloat16` matmul).
    `q`: `[B, H_q, S_q, D]`, `k`/`v`: `[B, H_kv, S_kv, D]`, `mask`: `[B, 1, S_q, S_kv]`.
    """
    k_rep = repeat_kv_jax(k, n_rep)
    v_rep = repeat_kv_jax(v, n_rep)
    scores = jnp.einsum("bhqd,bhkd->bhqk", q, k_rep) + mask
    probs = jax.nn.softmax(scores.astype(jnp.float32), axis=-1).astype(q.dtype)
    out = jnp.einsum("bhqk,bhkd->bhqd", probs, v_rep)
    B, H_q, S_q, D = out.shape
    return jnp.swapaxes(out, 1, 2).reshape(B, S_q, H_q * D)


def extract_layer_group_weights(
    layers: Sequence[torch.nn.Module],
    indices: Sequence[int],
    dev: Any,
    has_kv: bool,
    grid_shape: Optional[Tuple[int, ...]] = None,
) -> Dict[str, jax.Array]:
    """Stack weights for a homogeneous group of layers on CPU and transfer once to TPU."""
    fields = [
        ("input_norm_w", lambda m: m.input_layernorm.weight),
        ("post_attn_norm_w", lambda m: m.post_attention_layernorm.weight),
        ("pre_mlp_norm_w", lambda m: m.pre_feedforward_layernorm.weight),
        ("post_mlp_norm_w", lambda m: m.post_feedforward_layernorm.weight),
        ("post_ple_norm_w", lambda m: m.post_per_layer_input_norm.weight),
        ("q_proj_w", lambda m: m.self_attn.q_proj.weight),
        ("q_norm_w", lambda m: m.self_attn.q_norm.weight),
        ("o_proj_w", lambda m: m.self_attn.o_proj.weight),
        ("gate_proj_w", lambda m: m.mlp.gate_proj.weight),
        ("up_proj_w", lambda m: m.mlp.up_proj.weight),
        ("down_proj_w", lambda m: m.mlp.down_proj.weight),
        ("ple_gate_w", lambda m: m.per_layer_input_gate.weight),
        ("ple_proj_w", lambda m: m.per_layer_projection.weight),
        ("layer_scalar", lambda m: m.layer_scalar.reshape(())),
    ]
    if has_kv:
        fields.extend([
            ("k_proj_w", lambda m: m.self_attn.k_proj.weight),
            ("v_proj_w", lambda m: m.self_attn.v_proj.weight),
            ("k_norm_w", lambda m: m.self_attn.k_norm.weight),
        ])

    out: Dict[str, jax.Array] = {}
    for name, getter in fields:
        stacked_pt = torch.stack([getter(layers[i]).detach().cpu() for i in indices], dim=0)
        if grid_shape is not None:
            stacked_pt = stacked_pt.reshape(*grid_shape, *stacked_pt.shape[1:])
        out[name] = torch_bf16_to_jax(stacked_pt, dev)
    return out


class JaxStaticGemma4Engine:
    """
    Pure JAX/XLA Static-Graph Engine for Gemma 4 E2B-it on Cloud TPU v5e-1.
    """

    def __init__(self, pt_cond_model: torch.nn.Module, tokenizer: Any, jax_device: Optional[Any] = None):
        self.tokenizer = tokenizer
        self.dev = jax_device or jax.devices()[0]
        text_model = pt_cond_model.model.language_model
        cfg = text_model.config
        self.cfg = cfg

        self.hidden_size = int(cfg.hidden_size)                        # 1536
        self.ple_dim = int(cfg.hidden_size_per_layer_input)            # 256
        self.num_layers = int(cfg.num_hidden_layers)                   # 35
        self.num_kv_shared = int(cfg.num_kv_shared_layers)             # 20
        self.first_kv_shared = self.num_layers - self.num_kv_shared    # 15
        self.num_q_heads = int(cfg.num_attention_heads)                # 8
        self.slid_n_kv = int(cfg.per_layer_config[0].num_key_value_heads)
        self.slid_head_dim = int(cfg.per_layer_config[0].head_dim)
        self.slid_n_rep = self.num_q_heads // self.slid_n_kv
        self.glob_n_kv = int(cfg.per_layer_config[4].num_key_value_heads)
        self.glob_head_dim = int(cfg.per_layer_config[4].head_dim)
        self.glob_n_rep = self.num_q_heads // self.glob_n_kv
        self.eps = float(cfg.rms_norm_eps)                             # 1e-6
        self.sliding_window = int(cfg.sliding_window)                  # 512
        self.pad_token_id = int(getattr(cfg, "pad_token_id", 0) or 0)

        # Collect stop token IDs
        stop_ids = set()
        eos_cfg = getattr(cfg, "eos_token_id", None)
        if isinstance(eos_cfg, int):
            stop_ids.add(eos_cfg)
        elif isinstance(eos_cfg, (list, tuple)):
            stop_ids.update(int(x) for x in eos_cfg)
        for tok_str in ["<eos>", "<turn|>", "<end_of_turn>"]:
            tid = tokenizer.convert_tokens_to_ids(tok_str)
            if isinstance(tid, int) and tid >= 0 and tid != tokenizer.unk_token_id:
                stop_ids.add(tid)
        self.stop_ids_list = sorted(stop_ids)
        self.stop_ids_jax = jax.device_put(jnp.array(self.stop_ids_list, dtype=jnp.int32), self.dev)

        # Global & Top-level Weights on TPU
        self.embed_w = torch_bf16_to_jax(text_model.embed_tokens.weight, self.dev)  # [262144, 1536] (tied with lm_head)
        self.embed_scale = jax.device_put(
            jnp.asarray(float(text_model.embed_tokens.embed_scale.to(torch.bfloat16).item()), dtype=jnp.bfloat16),
            self.dev,
        )
        self.ple_embed_w = torch_bf16_to_jax(text_model.embed_tokens_per_layer.weight, self.dev)  # [262144, 8960]
        self.ple_embed_scale = jax.device_put(
            jnp.asarray(float(text_model.embed_tokens_per_layer.embed_scale.to(torch.bfloat16).item()), dtype=jnp.bfloat16),
            self.dev,
        )
        self.ple_model_proj_w = torch_bf16_to_jax(text_model.per_layer_model_projection.weight, self.dev)  # [8960, 1536]
        self.ple_model_proj_scale = jax.device_put(
            jnp.asarray(float(text_model.per_layer_model_projection_scale), dtype=jnp.bfloat16),
            self.dev,
        )
        self.ple_proj_norm_w = torch_bf16_to_jax(text_model.per_layer_projection_norm.weight, self.dev)  # [256]
        self.ple_input_scale = jax.device_put(
            jnp.asarray(float(text_model.per_layer_input_scale), dtype=jnp.bfloat16),
            self.dev,
        )
        self.final_norm_w = torch_bf16_to_jax(text_model.norm.weight, self.dev)  # [1536]

        # RoPE inverse frequencies
        rot = text_model.rotary_emb
        self.inv_freq_slid = torch_bf16_to_jax(rot.sliding_attention_inv_freq.float(), self.dev)  # [128] float32
        self.inv_freq_glob = torch_bf16_to_jax(rot.full_attention_inv_freq.float(), self.dev)     # [256] float32

        # Group the 35 layers into 4 homogeneous stacked tensor groups:
        z1_slid_idx = [l for l in range(15) if cfg.layer_types[l] == "sliding_attention"]
        z1_glob_idx = [l for l in range(15) if cfg.layer_types[l] == "full_attention"]
        z2_slid_idx = [l for l in range(15, 35) if cfg.layer_types[l] == "sliding_attention"]
        z2_glob_idx = [l for l in range(15, 35) if cfg.layer_types[l] == "full_attention"]

        self.z1_slid = extract_layer_group_weights(text_model.layers, z1_slid_idx, self.dev, has_kv=True, grid_shape=(3, 4))
        self.z1_glob = extract_layer_group_weights(text_model.layers, z1_glob_idx, self.dev, has_kv=True)
        self.z2_slid = extract_layer_group_weights(text_model.layers, z2_slid_idx, self.dev, has_kv=False, grid_shape=(4, 4))
        self.z2_glob = extract_layer_group_weights(text_model.layers, z2_glob_idx, self.dev, has_kv=False)

        self._orig_z2_slid_scalar = self.z2_slid["layer_scalar"]
        self._orig_z2_glob_scalar = self.z2_glob["layer_scalar"]
        self._orig_z2_slid_ple_proj = self.z2_slid["ple_proj_w"]
        self._orig_z2_glob_ple_proj = self.z2_glob["ple_proj_w"]

        self.z2_slid_active = jax.device_put(jnp.ones((4, 4), dtype=jnp.bool_), self.dev)
        self.z2_glob_active = jax.device_put(jnp.ones((4,), dtype=jnp.bool_), self.dev)

        self._compile_kernels()

    def restore_defaults(self) -> None:
        """Restore unmodified Zone-2 scalars, PLE projections, and active layer masks (0 recompiles!)."""
        self.z2_slid["layer_scalar"] = self._orig_z2_slid_scalar
        self.z2_glob["layer_scalar"] = self._orig_z2_glob_scalar
        self.z2_slid["ple_proj_w"] = self._orig_z2_slid_ple_proj
        self.z2_glob["ple_proj_w"] = self._orig_z2_glob_ple_proj
        self.z2_slid_active = jax.device_put(jnp.ones((4, 4), dtype=jnp.bool_), self.dev)
        self.z2_glob_active = jax.device_put(jnp.ones((4,), dtype=jnp.bool_), self.dev)

    def apply_layer_bypass(self, bypass_layers: Sequence[int], compensate_scalar: bool = True) -> None:
        """
        Configure dormant KV-shared layer bypass (`l >= 15`) with optional `layer_scalar` grafting.
        Triggers ZERO XLA recompiles because `z2_slid_active` and `layer_scalar` are dynamic array inputs!
        """
        self.restore_defaults()
        bypass_set = sorted(set(bypass_layers))
        slid_sc = np.array(self._orig_z2_slid_scalar.astype(jnp.float32), dtype=np.float32)  # [4, 4]
        glob_sc = np.array(self._orig_z2_glob_scalar.astype(jnp.float32), dtype=np.float32)  # [4]
        slid_act = np.ones((4, 4), dtype=bool)
        glob_act = np.ones((4,), dtype=bool)

        def get_sc(l_idx: int) -> float:
            b = (l_idx - 15) // 5
            rem = (l_idx - 15) % 5
            return float(glob_sc[b]) if rem == 4 else float(slid_sc[b, rem])

        def mult_sc(l_idx: int, factor: float) -> None:
            b = (l_idx - 15) // 5
            rem = (l_idx - 15) % 5
            if rem == 4:
                glob_sc[b] *= factor
            else:
                slid_sc[b, rem] *= factor

        if compensate_scalar:
            for l in bypass_set:
                lam = get_sc(l)
                prev = l - 1
                while prev in bypass_set and prev >= 15:
                    prev -= 1
                mult_sc(prev, lam)

        for l in bypass_set:
            b = (l - 15) // 5
            rem = (l - 15) % 5
            if rem == 4:
                glob_act[b] = False
            else:
                slid_act[b, rem] = False

        self.z2_slid["layer_scalar"] = jax.device_put(jnp.asarray(slid_sc, dtype=jnp.bfloat16), self.dev)
        self.z2_glob["layer_scalar"] = jax.device_put(jnp.asarray(glob_sc, dtype=jnp.bfloat16), self.dev)
        self.z2_slid_active = jax.device_put(jnp.asarray(slid_act, dtype=jnp.bool_), self.dev)
        self.z2_glob_active = jax.device_put(jnp.asarray(glob_act, dtype=jnp.bool_), self.dev)

    def apply_s1d_prose_subspace_surgery(self, prose_basis_pt: torch.Tensor) -> None:
        """
        Project out rank-8 conversational prose-filler basis `Q` (`[1536, 8]`) from `L25..L34`
        `per_layer_projection.weight` (`b = 2, 3` in Zone 2) on TPU with zero XLA recompilation!
        """
        q_jax = torch_bf16_to_jax(prose_basis_pt.float(), self.dev)  # [1536, 8] float32

        def proj_out(w_bf16: jax.Array) -> jax.Array:
            w_f32 = w_bf16.astype(jnp.float32)
            proj = jnp.einsum("dr,...dp->...rp", q_jax, w_f32)
            w_new = w_f32 - jnp.einsum("dr,...rp->...dp", q_jax, proj)
            return w_new.astype(jnp.bfloat16)

        slid_w = self._orig_z2_slid_ple_proj
        glob_w = self._orig_z2_glob_ple_proj
        self.z2_slid["ple_proj_w"] = slid_w.at[2:].set(proj_out(slid_w[2:]))
        self.z2_glob["ple_proj_w"] = glob_w.at[2:].set(proj_out(glob_w[2:]))

    def _compile_kernels(self) -> None:
        eps = self.eps

        def compute_embeddings_and_ple(
            input_ids: jax.Array,
            embed_w: jax.Array,
            embed_scale: jax.Array,
            ple_embed_w: jax.Array,
            ple_embed_scale: jax.Array,
            ple_model_proj_w: jax.Array,
            ple_model_proj_scale: jax.Array,
            ple_proj_norm_w: jax.Array,
            ple_input_scale: jax.Array,
        ) -> Tuple[jax.Array, jax.Array]:
            B, S = input_ids.shape
            inputs_embeds = jnp.take(embed_w, input_ids, axis=0) * embed_scale  # [B, S, 1536]
            ple_tok = jnp.take(ple_embed_w, input_ids, axis=0) * ple_embed_scale
            ple_tok = ple_tok.reshape(B, S, 35, 256)
            ple_proj = jnp.einsum("bsd,pd->bsp", inputs_embeds, ple_model_proj_w) * ple_model_proj_scale
            ple_proj = ple_proj.reshape(B, S, 35, 256)
            ple_proj = rms_norm(ple_proj, ple_proj_norm_w, eps)
            per_layer_inputs = (ple_proj + ple_tok) * ple_input_scale  # [B, S, 35, 256]
            return inputs_embeds, per_layer_inputs

        def compute_rope(pos_ids: jax.Array, inv_freq: jax.Array) -> Tuple[jax.Array, jax.Array]:
            freqs = pos_ids[..., None].astype(jnp.float32) * inv_freq[None, None, :]
            emb = jnp.concatenate([freqs, freqs], axis=-1)
            return jnp.cos(emb).astype(jnp.bfloat16), jnp.sin(emb).astype(jnp.bfloat16)

        def post_attn_mlp_ple(
            h: jax.Array,
            attn_out: jax.Array,
            ple_in: jax.Array,
            w: Dict[str, jax.Array],
        ) -> jax.Array:
            h = h + rms_norm(attn_out, w["post_attn_norm_w"], eps)
            r = h
            h_norm = rms_norm(h, w["pre_mlp_norm_w"], eps)
            gate = jax.nn.gelu(jnp.einsum("bsd,fd->bsf", h_norm, w["gate_proj_w"]), approximate=True)
            up = jnp.einsum("bsd,fd->bsf", h_norm, w["up_proj_w"])
            mlp_out = jnp.einsum("bsf,df->bsd", gate * up, w["down_proj_w"])
            h = r + rms_norm(mlp_out, w["post_mlp_norm_w"], eps)

            r_ple = h
            ple_g = jax.nn.gelu(jnp.einsum("bsd,pd->bsp", h, w["ple_gate_w"]), approximate=True)
            ple_u = jnp.einsum("bsp,dp->bsd", ple_g * ple_in, w["ple_proj_w"])
            h = r_ple + rms_norm(ple_u, w["post_ple_norm_w"], eps)
            return h * w["layer_scalar"]

        def run_active_layer(
            h: jax.Array,
            ple_in: jax.Array,
            cos: jax.Array,
            sin: jax.Array,
            mask: jax.Array,
            k_slot: jax.Array,
            v_slot: jax.Array,
            write_pos: jax.Array,
            w: Dict[str, jax.Array],
            head_dim: int,
            n_kv: int,
            n_rep: int,
            is_prefill: bool,
        ) -> Tuple[jax.Array, jax.Array, jax.Array]:
            B, S, _ = h.shape
            h_norm = rms_norm(h, w["input_norm_w"], eps)
            q = jnp.einsum("bsd,hd->bsh", h_norm, w["q_proj_w"]).reshape(B, S, self.num_q_heads, head_dim)
            k = jnp.einsum("bsd,hd->bsh", h_norm, w["k_proj_w"]).reshape(B, S, n_kv, head_dim)
            v = jnp.einsum("bsd,hd->bsh", h_norm, w["v_proj_w"]).reshape(B, S, n_kv, head_dim)

            q = apply_rope(rms_norm(q, w["q_norm_w"], eps), cos, sin)
            k = apply_rope(rms_norm(k, w["k_norm_w"], eps), cos, sin)
            v = rms_norm(v, None, eps)

            q_t = jnp.swapaxes(q, 1, 2)  # [B, 8, S, D]
            k_t = jnp.swapaxes(k, 1, 2)  # [B, n_kv, S, D]
            v_t = jnp.swapaxes(v, 1, 2)  # [B, n_kv, S, D]

            k_slot_new = jax.lax.dynamic_update_slice(k_slot, k_t, (0, 0, write_pos, 0))
            v_slot_new = jax.lax.dynamic_update_slice(v_slot, v_t, (0, 0, write_pos, 0))

            k_attn = k_t if is_prefill else k_slot_new
            v_attn = v_t if is_prefill else v_slot_new

            attn_vec = attention_core(q_t, k_attn, v_attn, mask, n_rep=n_rep)
            attn_out = jnp.einsum("bsd,od->bso", attn_vec, w["o_proj_w"])
            h_next = post_attn_mlp_ple(h, attn_out, ple_in, w)
            return h_next, k_slot_new, v_slot_new

        def run_shared_layer(
            h: jax.Array,
            ple_in: jax.Array,
            cos: jax.Array,
            sin: jax.Array,
            mask: jax.Array,
            k_shared: jax.Array,
            v_shared: jax.Array,
            w: Dict[str, jax.Array],
            is_active: jax.Array,
            head_dim: int,
            n_rep: int,
        ) -> jax.Array:
            def _do_layer(h_in: jax.Array) -> jax.Array:
                B, S, _ = h_in.shape
                h_norm = rms_norm(h_in, w["input_norm_w"], eps)
                q = jnp.einsum("bsd,hd->bsh", h_norm, w["q_proj_w"]).reshape(B, S, self.num_q_heads, head_dim)
                q = apply_rope(rms_norm(q, w["q_norm_w"], eps), cos, sin)
                q_t = jnp.swapaxes(q, 1, 2)
                attn_vec = attention_core(q_t, k_shared, v_shared, mask, n_rep=n_rep)
                attn_out = jnp.einsum("bsd,od->bso", attn_vec, w["o_proj_w"])
                return post_attn_mlp_ple(h_in, attn_out, ple_in, w)

            return jax.lax.cond(is_active, _do_layer, lambda h_in: h_in, h)

        def forward_35_layers(
            input_ids: jax.Array,
            pos_ids: jax.Array,
            mask_slid: jax.Array,
            mask_glob: jax.Array,
            write_pos: jax.Array,
            k_slid_cache: jax.Array,
            v_slid_cache: jax.Array,
            k_glob_cache: jax.Array,
            v_glob_cache: jax.Array,
            params: Dict[str, Any],
            is_prefill: bool,
        ) -> Tuple[jax.Array, jax.Array, jax.Array, jax.Array, jax.Array]:
            h, ple_all = compute_embeddings_and_ple(
                input_ids,
                params["embed_w"],
                params["embed_scale"],
                params["ple_embed_w"],
                params["ple_embed_scale"],
                params["ple_model_proj_w"],
                params["ple_model_proj_scale"],
                params["ple_proj_norm_w"],
                params["ple_input_scale"],
            )
            cos_slid, sin_slid = compute_rope(pos_ids, params["inv_freq_slid"])
            cos_glob, sin_glob = compute_rope(pos_ids, params["inv_freq_glob"])

            ple_t = jnp.moveaxis(ple_all, 2, 0)  # [35, B, S, 256]
            ple_z1 = ple_t[:15].reshape(3, 5, *ple_t.shape[1:])
            ple_z2 = ple_t[15:].reshape(4, 5, *ple_t.shape[1:])

            k_slid_3d = k_slid_cache.reshape(3, 4, *k_slid_cache.shape[1:])
            v_slid_3d = v_slid_cache.reshape(3, 4, *v_slid_cache.shape[1:])

            def z1_block_fn(h_curr, block_inp):
                w_slid_b, w_glob_b, ple_b, k_sb, v_sb, k_gb, v_gb = block_inp

                def z1_slid_fn(h_s, s_inp):
                    w_s, ple_s, ks, vs = s_inp
                    h_next, ks_new, vs_new = run_active_layer(
                        h_s, ple_s, cos_slid, sin_slid, mask_slid, ks, vs, write_pos,
                        w_s, head_dim=self.slid_head_dim, n_kv=self.slid_n_kv, n_rep=self.slid_n_rep, is_prefill=is_prefill
                    )
                    return h_next, (ks_new, vs_new)

                h_after_slid, (ks_out, vs_out) = jax.lax.scan(
                    z1_slid_fn, h_curr, (w_slid_b, ple_b[:4], k_sb, v_sb)
                )
                h_after_glob, kg_out, vg_out = run_active_layer(
                    h_after_slid, ple_b[4], cos_glob, sin_glob, mask_glob, k_gb, v_gb, write_pos,
                    w_glob_b, head_dim=self.glob_head_dim, n_kv=self.glob_n_kv, n_rep=self.glob_n_rep, is_prefill=is_prefill
                )
                return h_after_glob, (ks_out, vs_out, kg_out, vg_out)

            h, (k_slid_out, v_slid_out, k_glob_out, v_glob_out) = jax.lax.scan(
                z1_block_fn,
                h,
                (params["z1_slid"], params["z1_glob"], ple_z1, k_slid_3d, v_slid_3d, k_glob_cache, v_glob_cache),
            )
            k_slid_new = k_slid_out.reshape(12, *k_slid_cache.shape[1:])
            v_slid_new = v_slid_out.reshape(12, *k_slid_cache.shape[1:])
            k_glob_new = k_glob_out
            v_glob_new = v_glob_out

            P = input_ids.shape[1]
            k_anchor_slid = k_slid_new[11, :, :, :P, :] if is_prefill else k_slid_new[11]
            v_anchor_slid = v_slid_new[11, :, :, :P, :] if is_prefill else v_slid_new[11]
            k_anchor_glob = k_glob_new[2, :, :, :P, :] if is_prefill else k_glob_new[2]
            v_anchor_glob = v_glob_new[2, :, :, :P, :] if is_prefill else v_glob_new[2]

            def z2_block_fn(h_curr, block_inp):
                w_slid_b, w_glob_b, act_slid_b, act_glob_b, ple_b = block_inp

                def z2_slid_fn(h_s, s_inp):
                    w_s, act_s, ple_s = s_inp
                    h_next = run_shared_layer(
                        h_s, ple_s, cos_slid, sin_slid, mask_slid, k_anchor_slid, v_anchor_slid,
                        w_s, act_s, head_dim=self.slid_head_dim, n_rep=self.slid_n_rep
                    )
                    return h_next, None

                h_after_slid, _ = jax.lax.scan(
                    z2_slid_fn, h_curr, (w_slid_b, act_slid_b, ple_b[:4])
                )
                h_after_glob = run_shared_layer(
                    h_after_slid, ple_b[4], cos_glob, sin_glob, mask_glob, k_anchor_glob, v_anchor_glob,
                    w_glob_b, act_glob_b, head_dim=self.glob_head_dim, n_rep=self.glob_n_rep
                )
                return h_after_glob, None

            h, _ = jax.lax.scan(
                z2_block_fn,
                h,
                (params["z2_slid"], params["z2_glob"], params["z2_slid_active"], params["z2_glob_active"], ple_z2),
            )

            h_final = rms_norm(h, params["final_norm_w"], eps)
            last_h = h_final[:, -1:, :]
            logits = jnp.einsum("bsd,vd->bsv", last_h, params["embed_w"])
            next_tok = jnp.argmax(logits, axis=-1).astype(jnp.int32)
            return next_tok, k_slid_new, v_slid_new, k_glob_new, v_glob_new

        def _prefill_impl(
            input_ids: jax.Array,
            valid_start: jax.Array,
            prompt_lens: jax.Array,
            k_slid_cache: jax.Array,
            v_slid_cache: jax.Array,
            k_glob_cache: jax.Array,
            v_glob_cache: jax.Array,
            params: Dict[str, Any],
        ):
            B, P = input_ids.shape
            arange_p = jnp.arange(P, dtype=jnp.int32)
            q_idx = arange_p[None, None, :, None]
            k_idx = arange_p[None, None, None, :]
            vs_4d = valid_start[:, None, None, None]

            is_pad_q = q_idx < vs_4d
            pad_self = is_pad_q & (k_idx == q_idx)
            real_causal = (~is_pad_q) & (k_idx >= vs_4d) & (k_idx <= q_idx)

            allow_full = pad_self | real_causal
            allow_slid = pad_self | (real_causal & ((q_idx - k_idx) < self.sliding_window))

            zero_bf16 = jnp.asarray(0.0, dtype=jnp.bfloat16)
            neg_inf = jnp.asarray(-3.38953139e38, dtype=jnp.bfloat16)
            mask_glob = jnp.where(allow_full, zero_bf16, neg_inf)
            mask_slid = jnp.where(allow_slid, zero_bf16, neg_inf)

            pos_ids = jnp.maximum(arange_p[None, :] - valid_start[:, None], 0)
            write_pos = jnp.asarray(0, dtype=jnp.int32)

            return forward_35_layers(
                input_ids, pos_ids, mask_slid, mask_glob, write_pos,
                k_slid_cache, v_slid_cache, k_glob_cache, v_glob_cache,
                params, is_prefill=True
            )

        def _decode_step_impl(
            next_tok: jax.Array,
            cur_pos: jax.Array,
            step_offset: jax.Array,
            k_slid_cache: jax.Array,
            v_slid_cache: jax.Array,
            k_glob_cache: jax.Array,
            v_glob_cache: jax.Array,
            valid_start: jax.Array,
            prompt_lens: jax.Array,
            params: Dict[str, Any],
        ):
            M = k_slid_cache.shape[3]
            k_idx_m = jnp.arange(M, dtype=jnp.int32)[None, None, None, :]
            vs_4d = valid_start[:, None, None, None]
            cp_4d = cur_pos.reshape(1, 1, 1, 1)

            allow_full = (k_idx_m >= vs_4d) & (k_idx_m <= cp_4d)
            allow_slid = allow_full & ((cp_4d - k_idx_m) < self.sliding_window)

            zero_bf16 = jnp.asarray(0.0, dtype=jnp.bfloat16)
            neg_inf = jnp.asarray(-3.38953139e38, dtype=jnp.bfloat16)
            mask_glob = jnp.where(allow_full, zero_bf16, neg_inf)
            mask_slid = jnp.where(allow_slid, zero_bf16, neg_inf)

            pos_ids = prompt_lens[:, None] + step_offset.reshape(1, 1)
            return forward_35_layers(
                next_tok, pos_ids, mask_slid, mask_glob, cur_pos,
                k_slid_cache, v_slid_cache, k_glob_cache, v_glob_cache,
                params, is_prefill=False
            )

        @partial(jax.jit, static_argnums=(3,))
        def jit_prefill_and_decode_scan(
            input_ids: jax.Array,
            valid_start: jax.Array,
            prompt_lens: jax.Array,
            max_new_tokens: int,
            stop_ids_dev: jax.Array,
            params: Dict[str, Any],
        ):
            B, P = input_ids.shape
            G = max_new_tokens
            M = P + G

            k_slid_init = jnp.zeros((12, B, self.slid_n_kv, M, self.slid_head_dim), dtype=jnp.bfloat16)
            v_slid_init = jnp.zeros((12, B, self.slid_n_kv, M, self.slid_head_dim), dtype=jnp.bfloat16)
            k_glob_init = jnp.zeros((3, B, self.glob_n_kv, M, self.glob_head_dim), dtype=jnp.bfloat16)
            v_glob_init = jnp.zeros((3, B, self.glob_n_kv, M, self.glob_head_dim), dtype=jnp.bfloat16)

            first_tok, ks, vs, kg, vg = _prefill_impl(
                input_ids, valid_start, prompt_lens, k_slid_init, v_slid_init, k_glob_init, v_glob_init, params
            )
            finished_init = jnp.isin(first_tok, stop_ids_dev)
            step_positions = jnp.arange(P, P + G - 1, dtype=jnp.int32)
            step_offsets = jnp.arange(0, G - 1, dtype=jnp.int32)
            pad_tok_vec = jnp.full((B, 1), self.pad_token_id, dtype=jnp.int32)

            def scan_body(carry, step_inp):
                cur_tok, fin, k_s, v_s, k_g, v_g = carry
                cur_pos, step_off = step_inp
                cand_tok, k_s_new, v_s_new, k_g_new, v_g_new = _decode_step_impl(
                    cur_tok, cur_pos, step_off, k_s, v_s, k_g, v_g, valid_start, prompt_lens, params
                )
                next_t = jnp.where(fin, pad_tok_vec, cand_tok)
                fin_new = fin | jnp.isin(next_t, stop_ids_dev)
                return (next_t, fin_new, k_s_new, v_s_new, k_g_new, v_g_new), next_t

            _, gen_seq = jax.lax.scan(
                scan_body,
                (first_tok, finished_init, ks, vs, kg, vg),
                (step_positions, step_offsets),
            )
            gen_tokens = jnp.concatenate([first_tok, jnp.swapaxes(gen_seq.squeeze(-1), 0, 1)], axis=1)
            return gen_tokens

        self._jit_prefill = partial(jax.jit, donate_argnums=(3, 4, 5, 6))(_prefill_impl)
        self._jit_decode_step = partial(jax.jit, donate_argnums=(3, 4, 5, 6))(_decode_step_impl)
        self._jit_prefill_and_decode_scan = jit_prefill_and_decode_scan

    def _get_params_dict(self) -> Dict[str, Any]:
        return {
            "embed_w": self.embed_w,
            "embed_scale": self.embed_scale,
            "ple_embed_w": self.ple_embed_w,
            "ple_embed_scale": self.ple_embed_scale,
            "ple_model_proj_w": self.ple_model_proj_w,
            "ple_model_proj_scale": self.ple_model_proj_scale,
            "ple_proj_norm_w": self.ple_proj_norm_w,
            "ple_input_scale": self.ple_input_scale,
            "final_norm_w": self.final_norm_w,
            "inv_freq_slid": self.inv_freq_slid,
            "inv_freq_glob": self.inv_freq_glob,
            "z1_slid": self.z1_slid,
            "z1_glob": self.z1_glob,
            "z2_slid": self.z2_slid,
            "z2_glob": self.z2_glob,
            "z2_slid_active": self.z2_slid_active,
            "z2_glob_active": self.z2_glob_active,
        }

    def format_chat_prompts(
        self,
        user_prompts: Sequence[str],
        enable_thinking: bool = False,
        system_prompt: Optional[str] = None,
    ) -> List[str]:
        formatted = []
        for p in user_prompts:
            msgs = []
            if system_prompt:
                msgs.append({"role": "system", "content": system_prompt})
            msgs.append({"role": "user", "content": p})
            txt = self.tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
            )
            formatted.append(txt)
        return formatted

    def _allocate_kv_caches(self, B: int, M: int) -> Tuple[jax.Array, jax.Array, jax.Array, jax.Array]:
        k_slid = jax.device_put(jnp.zeros((12, B, self.slid_n_kv, M, self.slid_head_dim), dtype=jnp.bfloat16), self.dev)
        v_slid = jax.device_put(jnp.zeros((12, B, self.slid_n_kv, M, self.slid_head_dim), dtype=jnp.bfloat16), self.dev)
        k_glob = jax.device_put(jnp.zeros((3, B, self.glob_n_kv, M, self.glob_head_dim), dtype=jnp.bfloat16), self.dev)
        v_glob = jax.device_put(jnp.zeros((3, B, self.glob_n_kv, M, self.glob_head_dim), dtype=jnp.bfloat16), self.dev)
        return k_slid, v_slid, k_glob, v_glob

    def generate_batch_jax_scan(
        self,
        formatted_prompts: Sequence[str],
        max_new_tokens: int = 256,
        prompt_pad_bucket: int = 128,
    ) -> Dict[str, Any]:
        """
        Optimal Whole-Loop Compiled JAX Generation (Fused Prefill + `jax.lax.scan` Decode).
        Executes prefill and all decode steps inside a SINGLE XLA launch with zero host-loop overhead.
        """
        B = len(formatted_prompts)
        enc = [self.tokenizer.encode(t, add_special_tokens=False) for t in formatted_prompts]
        max_p_actual = max(len(ids) for ids in enc)
        P = max(prompt_pad_bucket, ((max_p_actual + prompt_pad_bucket - 1) // prompt_pad_bucket) * prompt_pad_bucket)
        G = max_new_tokens

        input_np = np.full((B, P), self.pad_token_id, dtype=np.int32)
        lens_np = np.zeros((B,), dtype=np.int32)
        for i, ids in enumerate(enc):
            if len(ids) > P:
                ids = ids[-P:]
            L = len(ids)
            input_np[i, P - L : P] = np.asarray(ids, dtype=np.int32)
            lens_np[i] = L

        params = self._get_params_dict()

        t_start = time.perf_counter()
        input_ids = jax.device_put(jnp.asarray(input_np), self.dev)
        prompt_lens = jax.device_put(jnp.asarray(lens_np), self.dev)
        valid_start = P - prompt_lens

        gen_tokens = self._jit_prefill_and_decode_scan(
            input_ids, valid_start, prompt_lens, G, self.stop_ids_jax, params
        )
        gen_tokens.block_until_ready()
        elapsed = time.perf_counter() - t_start

        gen_cpu = np.asarray(gen_tokens).tolist()
        stop_set = set(self.stop_ids_list)
        texts: List[str] = []
        tok_counts: List[int] = []
        for row in gen_cpu:
            trimmed = []
            for tid in row:
                if tid in stop_set:
                    break
                trimmed.append(tid)
            tok_counts.append(len(trimmed))
            texts.append(self.tokenizer.decode(trimmed, skip_special_tokens=False))

        total_gen_toks = sum(tok_counts)
        return {
            "texts": texts,
            "token_counts": tok_counts,
            "total_generated_tokens": total_gen_toks,
            "steps_executed": G,
            "batch_size": B,
            "prompt_pad_len": P,
            "elapsed_sec": round(elapsed, 4),
            "effective_tok_per_sec": round(total_gen_toks / max(elapsed, 1e-6), 2),
            "raw_step_tok_per_sec": round((B * G) / max(elapsed, 1e-6), 2),
        }

    def generate_batch_jax_step_loop(
        self,
        formatted_prompts: Sequence[str],
        max_new_tokens: int = 97,
        prompt_pad_bucket: int = 128,
    ) -> Dict[str, Any]:
        """
        Python-loop step-by-step `@jax.jit` with `dynamic_update_slice` (direct 1-to-1 JAX analogue
        to PyTorch's `StaticGemma4Engine` before fusing the decode loop with `jax.lax.scan`).
        """
        B = len(formatted_prompts)
        enc = [self.tokenizer.encode(t, add_special_tokens=False) for t in formatted_prompts]
        max_p_actual = max(len(ids) for ids in enc)
        P = max(prompt_pad_bucket, ((max_p_actual + prompt_pad_bucket - 1) // prompt_pad_bucket) * prompt_pad_bucket)
        G = max_new_tokens
        M = P + G

        input_np = np.full((B, P), self.pad_token_id, dtype=np.int32)
        lens_np = np.zeros((B,), dtype=np.int32)
        for i, ids in enumerate(enc):
            if len(ids) > P:
                ids = ids[-P:]
            L = len(ids)
            input_np[i, P - L : P] = np.asarray(ids, dtype=np.int32)
            lens_np[i] = L

        k_slid, v_slid, k_glob, v_glob = self._allocate_kv_caches(B, M)
        params = self._get_params_dict()

        t_start = time.perf_counter()
        input_ids = jax.device_put(jnp.asarray(input_np), self.dev)
        prompt_lens = jax.device_put(jnp.asarray(lens_np), self.dev)
        valid_start = P - prompt_lens

        next_tok, k_slid, v_slid, k_glob, v_glob = self._jit_prefill(
            input_ids, valid_start, prompt_lens, k_slid, v_slid, k_glob, v_glob, params
        )
        step_positions = jax.device_put(jnp.arange(P, P + G - 1, dtype=jnp.int32), self.dev)
        step_offsets = jax.device_put(jnp.arange(0, G - 1, dtype=jnp.int32), self.dev)

        tok_list = [next_tok]
        for s in range(G - 1):
            next_tok, k_slid, v_slid, k_glob, v_glob = self._jit_decode_step(
                next_tok, step_positions[s], step_offsets[s],
                k_slid, v_slid, k_glob, v_glob, valid_start, prompt_lens, params
            )
            tok_list.append(next_tok)

        gen_tokens = jnp.concatenate(tok_list, axis=1)
        gen_tokens.block_until_ready()
        elapsed = time.perf_counter() - t_start

        gen_cpu = np.asarray(gen_tokens).tolist()
        texts = [self.tokenizer.decode(row, skip_special_tokens=False) for row in gen_cpu]
        return {
            "texts": texts,
            "steps_executed": G,
            "batch_size": B,
            "elapsed_sec": round(elapsed, 4),
            "raw_step_tok_per_sec": round((B * G) / max(elapsed, 1e-6), 2),
        }

    def generate_dataset(
        self,
        user_prompts: Sequence[str],
        batch_size: int = 8,
        max_new_tokens: int = 256,
        prompt_pad_bucket: int = 256,
        enable_thinking: bool = False,
        system_prompt: Optional[str] = None,
    ) -> Dict[str, Any]:
        """
        Evaluates an entire dataset of prompts in uniform static batches using
        `generate_batch_jax_scan` (with right-padding stripped from tail batches).
        """
        formatted = self.format_chat_prompts(
            user_prompts, enable_thinking=enable_thinking, system_prompt=system_prompt
        )
        all_texts: List[str] = []
        all_counts: List[int] = []
        total_steps = 0
        total_elapsed = 0.0

        for start in range(0, len(formatted), batch_size):
            chunk = formatted[start : start + batch_size]
            actual_b = len(chunk)
            if actual_b < batch_size:
                chunk = list(chunk) + [chunk[-1]] * (batch_size - actual_b)

            res = self.generate_batch_jax_scan(
                chunk,
                max_new_tokens=max_new_tokens,
                prompt_pad_bucket=prompt_pad_bucket,
            )
            all_texts.extend(res["texts"][:actual_b])
            all_counts.extend(res["token_counts"][:actual_b])
            total_steps += res["steps_executed"]
            total_elapsed += res["elapsed_sec"]

        total_gen_toks = sum(all_counts)
        return {
            "texts": all_texts,
            "token_counts": all_counts,
            "total_generated_tokens": total_gen_toks,
            "total_steps": total_steps,
            "elapsed_sec": round(total_elapsed, 3),
            "effective_tok_per_sec": round(total_gen_toks / max(total_elapsed, 1e-6), 2),
            "mean_tokens_per_item": round(total_gen_toks / max(len(user_prompts), 1), 1),
        }


def run_jax_kv_update_microbenchmark(steps: int = 16) -> str:
    """
    16-step JAX KV-cache update microbenchmark on TPU/GPU comparing:
    1. Dynamic concatenation (jnp.concatenate)
    2. Static buffer with Python integer slice (.at[i:i+1].set)
    3. jax.lax.dynamic_update_slice with buffer donation in Python loop
    4. Fused jax.lax.scan + jax.lax.dynamic_update_slice (0 Python loop overhead)
    """
    dev = jax.devices()[0]
    q = jax.device_put(jnp.ones((1, 8, 1, 256), dtype=jnp.bfloat16), dev)
    k_steps = jax.device_put(jnp.ones((steps, 1, 1, 1, 256), dtype=jnp.bfloat16), dev)

    # Method 1: Dynamic jnp.concatenate (shape grows -> recompiles every step!)
    @jax.jit
    def step_concat(k_dyn, k_new, q_vec):
        k_next = jnp.concatenate([k_dyn, k_new], axis=2)
        attn = jax.nn.softmax(jnp.einsum("bhqd,bhkd->bhqk", q_vec, jnp.repeat(k_next, 8, axis=1)), axis=-1)
        return k_next, attn

    k_dyn = jax.device_put(jnp.zeros((1, 1, 4, 256), dtype=jnp.bfloat16), dev)
    t0 = time.perf_counter()
    for i in range(steps):
        k_dyn, attn = step_concat(k_dyn, k_steps[i], q)
    attn.block_until_ready()
    t_concat_ms = (time.perf_counter() - t0) * 1000.0

    # Method 2: Static buffer with Python int slice (static_argnums -> recompiles every step!)
    @partial(jax.jit, static_argnums=(2,))
    def step_py_slice(buf, k_new, idx_int, q_vec):
        buf_next = buf.at[:, :, idx_int : idx_int + 1, :].set(k_new)
        attn = jax.nn.softmax(jnp.einsum("bhqd,bhkd->bhqk", q_vec, jnp.repeat(buf_next, 8, axis=1)), axis=-1)
        return buf_next, attn

    buf_slice = jax.device_put(jnp.zeros((1, 1, 64, 256), dtype=jnp.bfloat16), dev)
    t0 = time.perf_counter()
    for i in range(steps):
        buf_slice, attn = step_py_slice(buf_slice, k_steps[i], i, q)
    attn.block_until_ready()
    t_slice_ms = (time.perf_counter() - t0) * 1000.0

    # Method 3: jax.lax.dynamic_update_slice + buffer donation in Python loop (0 recompiles!)
    @partial(jax.jit, donate_argnums=(0,))
    def step_dus(buf, k_new, pos_t, q_vec):
        buf_next = jax.lax.dynamic_update_slice(buf, k_new, (0, 0, pos_t, 0))
        attn = jax.nn.softmax(jnp.einsum("bhqd,bhkd->bhqk", q_vec, jnp.repeat(buf_next, 8, axis=1)), axis=-1)
        return buf_next, attn

    buf_dus = jax.device_put(jnp.zeros((1, 1, 64, 256), dtype=jnp.bfloat16), dev)
    pos_arr = jax.device_put(jnp.arange(steps, dtype=jnp.int32), dev)
    buf_dus, attn = step_dus(buf_dus, k_steps[0], pos_arr[0], q)  # Warmup 1 compile
    attn.block_until_ready()

    t0 = time.perf_counter()
    for i in range(steps):
        buf_dus, attn = step_dus(buf_dus, k_steps[i], pos_arr[i], q)
    attn.block_until_ready()
    t_dus_ms = (time.perf_counter() - t0) * 1000.0

    # Method 4: Fused jax.lax.scan + jax.lax.dynamic_update_slice (0 Python loop overhead!)
    @partial(jax.jit, donate_argnums=(0,))
    def scan_dus(buf, k_all, pos_all, q_vec):
        def body(carry_buf, xs):
            k_new, pos_t = xs
            b_next = jax.lax.dynamic_update_slice(carry_buf, k_new, (0, 0, pos_t, 0))
            attn = jax.nn.softmax(jnp.einsum("bhqd,bhkd->bhqk", q_vec, jnp.repeat(b_next, 8, axis=1)), axis=-1)
            return b_next, attn
        return jax.lax.scan(body, buf, (k_all, pos_all))

    buf_scan = jax.device_put(jnp.zeros((1, 1, 64, 256), dtype=jnp.bfloat16), dev)
    buf_scan, attn_seq = scan_dus(buf_scan, k_steps, pos_arr, q)  # Warmup 1 compile
    attn_seq.block_until_ready()

    t0 = time.perf_counter()
    buf_scan, attn_seq = scan_dus(buf_scan, k_steps, pos_arr, q)
    attn_seq.block_until_ready()
    t_scan_ms = (time.perf_counter() - t0) * 1000.0

    report = (
        f"1. JAX Dynamic Concat (jnp.concatenate)      | Compiles: 16 | Time: {t_concat_ms:7.2f} ms (1.0x)\n"
        f"2. JAX Python Slice (.at[i:i+1].set)         | Compiles: 16 | Time: {t_slice_ms:7.2f} ms ({t_concat_ms/t_slice_ms:.1f}x)\n"
        f"3. JAX dynamic_update_slice (Python loop)    | Compiles:  0 | Time: {t_dus_ms:7.2f} ms ({t_concat_ms/t_dus_ms:.1f}x)\n"
        f"4. JAX Fused lax.scan + dynamic_update_slice | Compiles:  0 | Time: {t_scan_ms:7.3f} ms ({t_concat_ms/t_scan_ms:.0f}x!)"
    )
    return report


In [ ]:
# @title 9. Run Pure JAX KV Microbenchmark & JaxStaticGemma4Engine (1,780.74 -> 1,849.65 tok/s)
import gc
import sys
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import importlib
import gemma4_surgery.benchmarks.jax_fast_infer as _jfi
importlib.reload(_jfi)
from gemma4_surgery.benchmarks.jax_fast_infer import run_jax_kv_update_microbenchmark, JaxStaticGemma4Engine

print("=== 16-Step JAX KV-Cache Update Microbenchmark ===")
print(run_jax_kv_update_microbenchmark())

if "MODEL" in globals() and MODEL is not None:
    MODEL_CPU = MODEL.cpu()
    gc.collect()
    JAX_ENGINE = JaxStaticGemma4Engine(MODEL_CPU, TOKENIZER)
    del MODEL_CPU
    gc.collect()

    smoke = ["Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?"]
    fmt = JAX_ENGINE.format_chat_prompts(smoke, enable_thinking=False)
    for b, max_new in [(1, 97), (8, 129), (16, 129), (32, 129)]:
        _ = JAX_ENGINE.generate_batch_jax_scan(fmt * b, max_new_tokens=max_new, prompt_pad_bucket=128)
        r_warm = JAX_ENGINE.generate_batch_jax_scan(fmt * b, max_new_tokens=max_new, prompt_pad_bucket=128)
        print(f"Batch={b:2d} | Warm Fused Scan: {r_warm['raw_step_tok_per_sec']:7.2f} tok/s ({r_warm['elapsed_sec']:.4f}s)")
